
# Unified astrocyte K-buffering model characterization notebook

This notebook consolidates the previous classifier/characterization scripts into one executable Google Colab pipeline for the minimal astrocyte potassium-buffering model.

## Scope

Primary goal: characterize the dynamical behavior of already simulated Optuna/SQLite databases in a transparent, mechanistic way. Biological validation is deliberately treated as secondary and provisional.

The notebook reads `*.db` Optuna SQLite files from `data.zip`, reconstructs hidden variables and currents by re-simulating accepted/top-ranked parameter sets, extracts membrane and extracellular potassium features, computes signed flux/current budgets, assigns mechanistic mode labels, applies corrected Zhou/Ma-inspired phenotype tags, and writes tables/plots for downstream inspection.

## Required input

A zip archive with the database layout used in your previous runs, for example:

```text
data.zip
└── data/
    ├── CONTROL_50nA.db ... CONTROL_175nA.db
    ├── MFA_50nA.db ... MFA_175nA.db
    ├── BARIUM_50nA.db ... BARIUM_175nA.db
    ├── CONTROL_TRACES.csv / MFA_TRACES.csv / BARIUM_TRACES.csv  # optional for this notebook
    └── *_BEST_FIT_PARAM.csv                                     # optional for this notebook
```

Optional input: `threshold_for_good_enough_fits.csv`. If it is absent, the notebook marks the selected top objective-ranked trials as **provisional accepted**. If a threshold CSV is present but rejects all trials in a sweep, the optional objective fallback keeps a clearly labelled provisional subset so the mechanism analysis does not collapse to an empty table.

## Key corrections encoded here

- `d` is **not** treated as anatomical syncytium size. It is retained as a Zhou/Ma-style coupling-strength or tightness scaling factor, while `Pkgap = pk*d` is the total GJ conductance/permeability proxy.
- `gs` and `alpha2 = gamma_s*Sigma_a/(w_a*F)` are retained as the reduced model's available spatial transfer surface-to-volume capacity.
- `chi_K`, `A_dKs = integral(|dKs_actual|)/integral(|dKs_if_open|)`, `Ks`, `alpha2*chi_K`, and `alpha2*A_dKs` are treated as dynamic recruited functional-syncytium/surface proxies, not literal anatomical `n`.
- Early raw `Jt/Jg` and `Js/Jg` ratios are superseded by signed flux budgets: local load `L = integral([dDKt/dt]_+)`, spatial export `S = integral([-dKs/dt]_+)`, and bath/source terms kept separately.
- Raw mean sigmoid is kept as a descriptive feature, but current-weighted dKs activation and endpoint/temporal recruitment classes are preferred for classification.
- Pump-related quantities are kept only as post hoc observables from `Ko` recovery/undershoot. They are **not** literal Na/K ATPase fluxes because the current model lacks an explicit pump state.

## Literature anchors used for interpretation

The notebook follows the modeling style of conductance-based ion-concentration models and electrodiffusive astrocyte/K-buffering models. It also uses the Ma/Zhou syncytial isopotentiality idea only as an interpretive analogue, not as direct validation. Useful anchors are Ma et al. 2016 on astrocyte gap-junction isopotentiality, Cressman et al. 2009 on sodium/potassium concentration dynamics, Halnes et al. 2013 on electrodiffusive astrocyte/neuron ion dynamics, and Odette & Newman 1988 on spatial potassium dynamics.

## Main outputs

All outputs are written under `OUT_DIR`:

- `tables/all_top_trial_features_and_acceptance.csv`
- `tables/accepted_mechanistic_scores_with_phenotypes.csv`
- `tables/Fv_membrane_feature_variability.json`
- `tables/Fk_Ko_feature_variability.json`
- `tables/feature_variability_Fv_Fk.csv`
- `tables/M_mode_vector_by_configuration.csv`
- `tables/M_mode_dictionary.json`
- `tables/measure_registry_status.csv`
- `tables/parameter_range_constraints_before_after.csv`
- `plots/modes/signed_flux_mode_quadrants_*.png`
- `plots/hidden_overlays/*.png` when enabled
- `run_summary.json`

## Runtime profiles

The default profile below is a **smoke run** so the notebook can be tested quickly in Colab. For the full analysis, set `RUN_PROFILE = "full"` in the configuration cell. The publication-scale settings are computationally heavier because every accepted/top-ranked trial is re-simulated.


## Implementation

The following cell defines the full analysis pipeline. It is self-contained and does not import the previous scripts or Optuna.

In [1]:
#!/usr/bin/env python3
"""
Buffering phenotype analysis for the minimal astrocytic K+ buffering model.

Purpose
-------
Classify accepted Optuna configurations into interpretable buffering phenotypes
linked to experimental observables:

- Kir strength / Ba2+ sensitivity       -> gki and integrated I_Kir
- coupling strength / Zhou s proxy     -> d and Pkgap = pk * d
- total GJ conductance / MFA effect     -> Pkgap and integrated I_kgap
- dynamic recruited syncytium proxy      -> chi_K / dKs activation / dynamic conversion factor
- low-range vs long-range redistribution -> dKs / sigmoid activation end-state and window activation
- stimulation load                      -> epsilon * (K_bath_middle - K_o0)
- buffering performance                 -> K_o peak, K_o AUC, clearance/source ratio
- gamma_s / alpha2 available surface     -> gs and alpha2 as available anatomical/surface-to-volume capacity proxies
- recruited surface capacity             -> alpha2 * chi_K or alpha2 * A_dKs
- K+ pump proxy                          -> K_o undershoot, post-stimulus clearance demand, pump-drive proxy
- MFA-Control contrast                   -> per-current/window delta and fold-change summaries

The script reads Optuna SQLite DBs directly, so Optuna does not need to be installed.
It can run on the supplied old data.zip layout, or on an extracted data directory.

Typical usage
-------------
python buffering_phenotype_pipeline.py \
  --data-zip data.zip \
  --out buffering_phenotype_outputs \
  --experiments CONTROL,MFA,BARIUM \
  --currents 50,75,100,125,150,175 \
  --top-n 300 \
  --sim-dt-ms 5 \
  --make-zip

If you have the control-derived threshold CSV:
python buffering_phenotype_pipeline.py \
  --data-zip data.zip \
  --threshold-csv threshold_for_good_enough_fits.csv \
  --out buffering_phenotype_outputs \
  --top-n 300 \
  --sim-dt-ms 5

Notes
-----
The key model-specific activation measure is:

    ionic_activation = ∫ |dKs_actual| dt / ∫ |dKs_if_gate_fully_open| dt

where dKs_if_gate_fully_open is computed by replacing the dimensionless transfer
activation factor by 1 while keeping the same voltage/current trajectory. This is
more robust than a raw time-average of the sigmoid, because it answers the mechanistic
question: "What fraction of the available junctional K-redistribution capacity was
actually recruited during this configuration/window?"

Final biophysical interpretation implemented here:
    d                 -> Zhou-style coupling strength s_model, not syncytium size n
    pk*d = Pkgap      -> total GJ conductance/permeability proxy
    chi_K(t)          -> state-dependent functional recruitment fraction
    gs/alpha2         -> available anatomical/surface-to-volume portion of syncytium
    alpha2*DKa*chi_K  -> dynamically recruited surface/conversion factor for spatial ionic flux
    n_func proxies    -> chi_K, dKs activation, and Ks state variables
    r_model           -> local ionic contribution vs syncytial voltage-clamp proxy
"""

from __future__ import annotations

import argparse
import json
import math
import os
import re
import shutil
import sqlite3
import sys
import tempfile
import warnings
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy import stats

try:
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# -----------------------------------------------------------------------------
# Experiment constants copied from the original notebook/preprint model
# -----------------------------------------------------------------------------
CURRENT_DICT_COLUMNS = {"50": 1, "75": 2, "100": 3, "125": 4, "150": 5, "175": 6}
CURRENT_DICT_K_BATH_VALUES = {
    "50": [4.8, 6.4, 4.8],
    "75": [4.8, 7.23, 4.8],
    "100": [4.8, 8.2, 4.8],
    "125": [4.8, 9.5, 4.8],
    "150": [4.8, 10.1, 4.8],
    "175": [4.8, 10.5, 4.8],
}
EXPERIMENT_K_BATH_TIME_MS = {
    "CONTROL": [0.0, 11173.0, 31173.0],
    "MFA": [0.0, 21140.0, 41140.0],
    "BARIUM": [0.0, 21140.0, 41140.0],
}
EXPERIMENT_CUTS = {"CONTROL": [0, 0.1], "MFA": [30, 0.2], "BARIUM": [30, 0.2]}
DEFAULT_Z0 = [-89.0, 0.0, 0.0, 0.0]
PENALTY_VALUE = 1.0e7
EPS = 1.0e-12
DEFAULT_DOMINANCE_MARGIN = 1.20
WINDOWS = ["M0", "M_rise", "M_decay", "M_tot"]


ORDERED_PARAM_KEYS = [
    "eps", "eps_middle", "gki", "pk", "gs", "gt", "ca", "d", "w_a",
    "wo", "wo_middle", "K_bath_value_middle", "gl_a", "Va_s", "Va_l",
    "switching_function", "zth", "zs", "hill_coefficient", "K_d",
]

FEATURES_FOR_FILTER = [
    "peak_depolarization_mV",
    "rise_slope_mV_per_s",
    "rise_tau_s",
    "plateau_slope_mV_per_s",
    "decay_slope_mV_per_s",
    "decay_tau_s",
    "undershoot_magnitude_mV",
    "return_slope_mV_per_s",
]

MEASURE_REGISTRY_ROWS = [
    {"measure_family": "state reconstruction", "measure": "Ko, Ka, DKt, Ks, Kg, DKa, EKa, currents", "status": "primary", "rationale": "Hidden-variable reconstruction is retained for sanity filtering and mechanism interpretation."},
    {"measure_family": "Vm/Ko features", "measure": "Fv and Fk trace-feature dictionaries", "status": "primary", "rationale": "Parallel Vm and extracellular K shape features are retained for fit filtering and variability summaries."},
    {"measure_family": "flux budget", "measure": "L=integral([dDKt/dt]+), S=integral([-dKs/dt]+), bath source/sink", "status": "primary", "rationale": "Signed local load and spatial export resolve the sign ambiguity in early Jt/Js ratios."},
    {"measure_family": "mechanistic mode", "measure": "D_F and D_I_elec with finite dominance margin", "status": "primary", "rationale": "Continuous axes are primary; strict/mixed labels are derived summaries, not biological ground truth."},
    {"measure_family": "recruitment", "measure": "dKs_activation = integral(|dKs_actual|)/integral(|dKs_if_open|)", "status": "primary", "rationale": "Current-weighted recruitment is more mechanistic than a raw mean gate value."},
    {"measure_family": "Zhou/Ma mapping", "measure": "d -> s_model; Pkgap=pk*d -> GJ conductance", "status": "primary", "rationale": "Corrected mapping; d is not syncytium size when pk also varies."},
    {"measure_family": "available surface", "measure": "gs and alpha2 = gamma_s*Sigma_a/(w_a*F)", "status": "primary", "rationale": "Interpreted as available spatial transfer surface-to-volume capacity in the reduced model."},
    {"measure_family": "functional syncytium", "measure": "chi_K, A_dKs, Ks, alpha2*chi_K, alpha2*A_dKs", "status": "primary", "rationale": "Dynamic recruited fraction/capacity; not anatomical n."},
    {"measure_family": "isopotentiality proxy", "measure": "r_model=|Va-Vs|/(|EKa-Vs|+eps), 1/(1+r_model)", "status": "secondary", "rationale": "Useful reduced-model analogue because the ODE has no explicit neighboring-cell voltage."},
    {"measure_family": "pump proxy", "measure": "Ko undershoot/recovery AUC proxies", "status": "secondary", "rationale": "Retained as observable recovery descriptors, explicitly not literal Na/K ATPase flux."},
    {"measure_family": "deprecated", "measure": "d as anatomical syncytium size n", "status": "deprecated", "rationale": "Superseded by d as coupling-strength proxy and gs/alpha2 plus chi/A_dKs as available/recruited surface."},
    {"measure_family": "deprecated", "measure": "raw Jt/Jg and Js/Jg without sign handling", "status": "deprecated", "rationale": "Superseded by signed flux-budget axes with local uptake and spatial export directionality."},
    {"measure_family": "deprecated", "measure": "raw mean sigmoid alone as syncytium size", "status": "superseded", "rationale": "Mean gate is retained as a descriptive feature, but current-weighted A_dKs and endpoint classes are preferred."},
]

# -----------------------------------------------------------------------------
# Data classes
# -----------------------------------------------------------------------------
@dataclass
class TrialSimulationContext:
    experiment_type: str
    current_na: int
    study_name: str
    objective_loss_type: str
    target_mean_mode: str
    t_ms: np.ndarray
    windows_s: Dict[str, Tuple[float, float]]
    fixed_params: Dict[str, object]


# -----------------------------------------------------------------------------
# Basic helpers
# -----------------------------------------------------------------------------
def safe_float(x: object, default: float = np.nan) -> float:
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default


def parse_csv_list(value: str, cast=str) -> List:
    if value is None or str(value).strip() == "":
        return []
    return [cast(v.strip()) for v in str(value).split(",") if v.strip()]


def ensure_data_dir(data_zip: Optional[Path], data_dir: Optional[Path], out_dir: Path) -> Path:
    if data_dir is not None and data_dir.exists():
        if (data_dir / "data").exists():
            return data_dir / "data"
        return data_dir
    if data_zip is None or not data_zip.exists():
        raise FileNotFoundError("Provide --data-zip or --data-dir")
    extract_dir = out_dir / "_extracted_data"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(data_zip, "r") as zf:
        zf.extractall(extract_dir)
    if (extract_dir / "data").exists():
        return extract_dir / "data"
    return extract_dir


def infer_study_name_from_db(db_path: Path) -> str:
    with sqlite3.connect(db_path) as con:
        df = pd.read_sql_query("SELECT study_name FROM studies ORDER BY study_id LIMIT 1", con)
    if df.empty:
        return db_path.stem
    return str(df.loc[0, "study_name"])


def infer_metadata_from_name(name: str, fallback_path: Optional[Path] = None) -> Tuple[str, int, str, str]:
    source = name or (fallback_path.stem if fallback_path else "")
    m = re.search(r"(CONTROL|MFA|BARIUM)[_\-](\d+)nA", source, flags=re.I)
    if not m:
        raise ValueError(f"Cannot infer experiment/current from {source}")
    experiment = m.group(1).upper()
    current = int(m.group(2))

    target_mean_mode = "centered" if "centered" in source else "default"
    if "centered_scaled" in source:
        target_mean_mode = "centered_scaled"
    objective_loss_type = "L2"
    for token in ["COMBINED", "HUBER", "LOG_COSH", "L1", "L2"]:
        if re.search(rf"(^|_)({token})(_|$)", source, flags=re.I):
            objective_loss_type = token.upper()
            break
    return experiment, current, target_mean_mode, objective_loss_type


def normalize_flat_params(params: Mapping[str, object]) -> Dict[str, object]:
    p = dict(params)
    aliases = {"Zth": "zth", "Zs": "zs", "Z_th": "zth", "Z_s": "zs", "w_o_middle": "wo_middle"}
    for old, new in aliases.items():
        if old in p and new not in p:
            p[new] = p[old]
    p.setdefault("wo_middle", 1.0)
    p.setdefault("eps_middle", 1.0)
    p.setdefault("w_a", 2000.0)
    p.setdefault("switching_function", "sigmoid")
    return p


def load_fixed_params_from_db(db_path: Path) -> Dict[str, object]:
    with sqlite3.connect(db_path) as con:
        df = pd.read_sql_query(
            """
            SELECT value_json
            FROM trial_system_attributes
            WHERE key='fixed_params'
            ORDER BY trial_id ASC
            LIMIT 1
            """,
            con,
        )
    if df.empty:
        return {}
    return normalize_flat_params(json.loads(df.loc[0, "value_json"]))


def load_search_space_from_db(db_path: Path) -> pd.DataFrame:
    with sqlite3.connect(db_path) as con:
        df = pd.read_sql_query(
            """
            SELECT param_name, distribution_json
            FROM trial_params
            GROUP BY param_name, distribution_json
            ORDER BY param_name
            """,
            con,
        )
    rows = []
    for _, row in df.iterrows():
        try:
            parsed = json.loads(row["distribution_json"])
            attrs = parsed.get("attributes", {})
        except Exception:
            parsed, attrs = {}, {}
        rows.append({
            "param_name": row["param_name"],
            "distribution_name": parsed.get("name", "unknown"),
            "low": attrs.get("low", np.nan),
            "high": attrs.get("high", np.nan),
            "log": attrs.get("log", np.nan),
            "step": attrs.get("step", np.nan),
        })
    return pd.DataFrame(rows)


def load_trials_from_db(db_path: Path, include_penalty: bool = False) -> pd.DataFrame:
    fixed = load_fixed_params_from_db(db_path)
    study_name = infer_study_name_from_db(db_path)
    experiment, current, target_mean_mode, objective_loss_type = infer_metadata_from_name(study_name, db_path)
    with sqlite3.connect(db_path) as con:
        trials = pd.read_sql_query(
            """
            SELECT t.trial_id, t.number AS trial_number, t.state, v.value AS objective
            FROM trials t
            LEFT JOIN trial_values v ON v.trial_id=t.trial_id AND v.objective=0
            ORDER BY t.number
            """,
            con,
        )
        params = pd.read_sql_query("SELECT trial_id, param_name, param_value FROM trial_params", con)
    trials = trials[trials["state"].astype(str).str.upper() == "COMPLETE"].copy()
    if params.empty:
        wide = pd.DataFrame(index=trials["trial_id"])
    else:
        wide = params.pivot_table(index="trial_id", columns="param_name", values="param_value", aggfunc="last")
    merged = trials.merge(wide, left_on="trial_id", right_index=True, how="left")

    rows = []
    all_keys = list(ORDERED_PARAM_KEYS)
    for k in sorted(params["param_name"].unique().tolist() if not params.empty else []):
        if k not in all_keys:
            all_keys.append(k)
    for _, r in merged.iterrows():
        objective = safe_float(r.get("objective"))
        if (not include_penalty) and ((not np.isfinite(objective)) or objective >= PENALTY_VALUE):
            continue
        p = dict(fixed)
        for key in all_keys:
            if key in r.index and pd.notna(r[key]):
                p[key] = r[key]
        p = normalize_flat_params(p)
        out = {
            "db_file": db_path.name,
            "study_name": study_name,
            "experiment": experiment,
            "current_na": current,
            "target_mean_mode": target_mean_mode,
            "objective_loss_type": objective_loss_type,
            "trial_id": int(r["trial_id"]),
            "trial_number": int(r["trial_number"]),
            "objective": objective,
        }
        for key in all_keys:
            out[key] = p.get(key, np.nan)
        out.update(derived_param_columns(p))
        rows.append(out)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values(["objective", "trial_number"], ascending=[True, True]).reset_index(drop=True)


def derived_param_columns(params: Mapping[str, object]) -> Dict[str, float]:
    p = normalize_flat_params(params)
    d = safe_float(p.get("d"))
    pk = safe_float(p.get("pk"))
    gki = safe_float(p.get("gki"))
    eps = safe_float(p.get("eps"))
    eps_mid = safe_float(p.get("eps_middle", 1.0))
    kb = safe_float(p.get("K_bath_value_middle"))
    wo = safe_float(p.get("wo"))
    wa = safe_float(p.get("w_a", 2000.0))
    return {
        "Pkgap": d * pk,
        "deprecated_dye_coupling_proxy_d_do_not_use": d,
        "junction_permeability_pk": pk,
        "kir_proxy_gki": gki,
        "wa_over_wo": wa / wo if np.isfinite(wa) and np.isfinite(wo) and wo != 0 else np.nan,
        "drive_Kbath_minus_Ko0": kb - 4.8 if np.isfinite(kb) else np.nan,
        "drive_eps_x_deltaKbath": eps * eps_mid * max(kb - 4.8, 0.0) if np.isfinite(eps) and np.isfinite(kb) else np.nan,
        "drive_eps_x_Kbath": eps * eps_mid * kb if np.isfinite(eps) and np.isfinite(kb) else np.nan,
    }


# -----------------------------------------------------------------------------
# Model equations and vectorized components
# -----------------------------------------------------------------------------
def build_paramdict(experiment: str, current_na: int, flat_params: Mapping[str, object]) -> Dict[str, Dict[str, object]]:
    p = normalize_flat_params(flat_params)
    current_key = str(int(current_na))
    paramdict = {
        "Astrocyte": {
            "Cm_a": float(p.get("ca", 400.0)),
            "g_kir": float(p.get("gki", 1.0)),
            "g_k_a": 0.0,
            "w_a": float(p.get("w_a", 2000.0)),
            "P_k": float(p.get("pk", 3e-5)),
            "A": 1.0,
            "gl_a": float(p.get("gl_a", 0.01)),
            "Va_l": float(p.get("Va_l", -70.0)),
            "Va_s": float(p.get("Va_s", -90.0)),
            "d_gap": float(p.get("d", 1.0)),
            "Va_0": -89.0,
            "Sig_a": 1600.0,
            "K_a0": 135.0,
            "F": 96485.0,
            "R": 8.314,
            "T": 298.0,
            "gama_t": float(p.get("gt", 6.0)),
            "gama_s": float(p.get("gs", 6.5)),
            "Z_s": float(p.get("zs", 0.05)) if p.get("zs", None) is not None else None,
            "Z_th": float(p.get("zth", 0.2)) if p.get("zth", None) is not None else None,
            "switching_function": str(p.get("switching_function", "sigmoid")),
        },
        "external": {
            "K_o0": 4.8,
            "w_o": float(p.get("wo", 1500.0)),
            "w_o_middle": float(p.get("wo_middle", 1.0)),
            "epsilon": float(p.get("eps", 1e-3)),
            "epsilon_middle": float(p.get("eps_middle", 1.0)),
            "K_bath": {
                "time": np.array(EXPERIMENT_K_BATH_TIME_MS[experiment], dtype=float),
                "value": CURRENT_DICT_K_BATH_VALUES[current_key].copy(),
            },
        },
    }
    paramdict["external"]["K_bath"]["value"][1] = float(
        p.get("K_bath_value_middle", paramdict["external"]["K_bath"]["value"][1])
    )
    if paramdict["Astrocyte"]["switching_function"] == "hill":
        paramdict["Astrocyte"]["hill_coefficient"] = float(p.get("hill_coefficient", 2.0))
        paramdict["Astrocyte"]["K_d"] = float(p.get("K_d", 1.0))
    return paramdict


def model(z: Sequence[float], t_ms: float, paramdict: Dict[str, Dict[str, object]]) -> List[float]:
    c = single_components(z, t_ms, paramdict)
    return [c["dVa"], c["dDKt"], c["dKs"], c["dKg"]]


def single_components(z: Sequence[float], t_ms: float, paramdict: Dict[str, Dict[str, object]]) -> Dict[str, float]:
    A = paramdict["Astrocyte"]
    E = paramdict["external"]
    times = np.asarray(E["K_bath"]["time"], dtype=float)
    values = np.asarray(E["K_bath"]["value"], dtype=float)
    idx = int(np.searchsorted(times, t_ms, side="right") - 1)
    idx = max(0, min(idx, len(values) - 1))
    K_bath = float(values[idx])
    eps = float(E["epsilon"]) * (float(E.get("epsilon_middle", 1.0)) if idx == 1 else 1.0)
    w_o = float(E["w_o"]) * (float(E.get("w_o_middle", 1.0)) if idx == 1 else 1.0)

    Va, DKt, Ks, Kg = [float(v) for v in z]
    DKa = DKt + Ks
    Ka = float(A["K_a0"]) + DKa
    Ko = float(E["K_o0"]) - (float(A["w_a"]) / w_o) * DKt + Kg
    K_ratio = Ko / Ka if Ka != 0 else 1e-8
    if K_ratio <= 0 or not np.isfinite(K_ratio):
        K_ratio = 1e-8
    EKa = 25.7 * math.log(K_ratio)

    with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
        Ika = float(A["g_k_a"]) * (Va - EKa)
        IKir = float(A["g_kir"]) * math.sqrt(abs(Ko)) * (Va - EKa) * (1.0 / (1.0 + math.exp((Va - EKa) / 19.2)))
        PH = 0.04 * (Va - float(A["Va_s"]))
        exp_neg = math.exp(-PH)
        denom = -1.0 + exp_neg
        if abs(denom) < 1e-12:
            denom = 1e-12 if denom >= 0 else -1e-12
        Pkgap = float(A["d_gap"]) * float(A["P_k"])
        Ikgap = Pkgap * float(A["F"]) * PH * (1.0 / denom) * ((Ka * exp_neg) - float(A["K_a0"]))
        Il = float(A["gl_a"]) * (Va - float(A["Va_l"]))

        sf = str(A.get("switching_function", "sigmoid"))
        Zth = A.get("Z_th", None)
        Zs = A.get("Z_s", None)
        if sf == "sigmoid":
            gate = 1.0 / (1.0 + math.exp((float(Zth) - DKt) * float(Zs)))
            Ths = DKa * gate
        elif sf == "tanh":
            gate = 0.5 * (1.0 + math.tanh((DKt - float(Zth)) * float(Zs)))
            Ths = DKa * gate
        elif sf == "hill":
            n = float(A.get("hill_coefficient", 2.0))
            Kd = float(A.get("K_d", 1.0))
            x = max(DKt, 0.0)
            gate = (x ** n) / (Kd ** n + x ** n + EPS)
            Ths = DKa * gate
        else:
            gate = 1.0
            Ths = DKa

        dVa = -(IKir + Ika + Il + Ikgap) / float(A["Cm_a"])
        dDKt = -(float(A["gama_t"]) * float(A["Sig_a"]) / (float(A["w_a"]) * float(A["F"]))) * (IKir + Ika)
        dKs = -Ths * (float(A["gama_s"]) * float(A["Sig_a"]) / (float(A["w_a"]) * float(A["F"]))) * Ikgap
        dKs_if_open = -DKa * (float(A["gama_s"]) * float(A["Sig_a"]) / (float(A["w_a"]) * float(A["F"]))) * Ikgap
        dKg = eps * (K_bath - Ko)

    return {
        "Va": Va, "DKt": DKt, "Ks": Ks, "Kg": Kg, "DKa": DKa, "Ka": Ka, "Ko": Ko,
        "EKa": EKa, "I_Kir": IKir, "I_k": Ika, "I_kgap": Ikgap, "I_l": Il,
        "I_total_net": IKir + Ika + Ikgap + Il,
        "gate_activation": gate, "Ths": Ths, "dVa": dVa, "dDKt": dDKt,
        "dKs": dKs, "dKs_if_open": dKs_if_open, "dKg": dKg, "dKo": (-(float(A["w_a"]) / max(float(w_o), EPS)) * dDKt + dKg), "K_bath": K_bath,
        "eps_effective": eps, "w_o_effective": w_o,
    }


def components_arrays(z: np.ndarray, t_ms: np.ndarray, paramdict: Dict[str, Dict[str, object]]) -> Dict[str, np.ndarray]:
    keys = [
        "Va", "DKt", "Ks", "Kg", "DKa", "Ka", "Ko", "EKa", "I_Kir", "I_k", "I_kgap", "I_l",
        "I_total_net", "gate_activation", "Ths", "dVa", "dDKt", "dKs", "dKs_if_open", "dKg", "dKo",
        "K_bath", "eps_effective", "w_o_effective",
    ]
    out = {k: np.empty(len(t_ms), dtype=float) for k in keys}
    out["t_ms"] = np.asarray(t_ms, dtype=float)
    out["t_s"] = np.asarray(t_ms, dtype=float) / 1000.0
    for i, (zi, ti) in enumerate(zip(z, t_ms)):
        c = single_components(zi, float(ti), paramdict)
        for k in keys:
            out[k][i] = c[k]
    return out


def make_context(experiment: str, current_na: int, study_name: str, objective_loss_type: str, target_mean_mode: str, sim_dt_ms: float, fixed_params: Dict[str, object]) -> TrialSimulationContext:
    # Simulation duration matches the original trace durations used in the notebook.
    if experiment == "CONTROL":
        end_ms = 50030.0
    else:
        end_ms = 72994.0
    t_ms = np.arange(0.0, end_ms + sim_dt_ms, sim_dt_ms)
    stim_start_s = EXPERIMENT_K_BATH_TIME_MS[experiment][1] / 1000.0
    stim_end_s = EXPERIMENT_K_BATH_TIME_MS[experiment][2] / 1000.0
    windows = {
        "M0": (0.0, stim_start_s),
        "M_rise": (stim_start_s, stim_end_s),
        "M_decay": (stim_end_s, float(t_ms[-1]) / 1000.0),
        "M_tot": (0.0, float(t_ms[-1]) / 1000.0),
    }
    return TrialSimulationContext(
        experiment_type=experiment,
        current_na=int(current_na),
        study_name=study_name,
        objective_loss_type=objective_loss_type,
        target_mean_mode=target_mean_mode,
        t_ms=t_ms,
        windows_s=windows,
        fixed_params=fixed_params,
    )


def simulate_trial(row: Mapping[str, object], context: TrialSimulationContext) -> Tuple[np.ndarray, Dict[str, np.ndarray], Dict[str, Dict[str, object]]]:
    params = {k: row[k] for k in ORDERED_PARAM_KEYS if k in row and pd.notna(row[k])}
    paramdict = build_paramdict(context.experiment_type, context.current_na, params)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        z = odeint(model, DEFAULT_Z0, context.t_ms, args=(paramdict,), mxstep=5000)
    if not np.isfinite(z).all():
        raise ValueError("nonfinite ODE solution")
    details = components_arrays(z, context.t_ms, paramdict)
    return z, details, paramdict


# -----------------------------------------------------------------------------
# Feature extraction and accepted-trial filtering
# -----------------------------------------------------------------------------
def moving_average(x: np.ndarray, window_pts: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    n = int(max(3, window_pts))
    if n % 2 == 0:
        n += 1
    return np.convolve(x, np.ones(n) / n, mode="same")


def extract_trace_features(t_s: np.ndarray, y_raw: np.ndarray, onset_s: float, offset_s: float) -> Dict[str, float]:
    t = np.asarray(t_s, dtype=float)
    y_raw = np.asarray(y_raw, dtype=float)
    if len(t) < 10:
        return {k: np.nan for k in FEATURES_FOR_FILTER}
    dt = np.nanmedian(np.diff(t))
    y = moving_average(y_raw, max(3, int(round(0.025 / max(dt, EPS)))))
    baseline_mask = (t >= max(t[0], onset_s - 5.0)) & (t < onset_s - 1.0)
    if baseline_mask.sum() < 3:
        baseline_mask = t < onset_s
    baseline = float(np.nanmedian(y_raw[baseline_mask])) if baseline_mask.sum() else float(y_raw[0])
    stim = (t >= onset_s) & (t <= offset_s)
    post = t >= offset_s
    if stim.sum() < 3 or post.sum() < 3:
        return {k: np.nan for k in FEATURES_FOR_FILTER}

    peak_i = np.where(stim)[0][np.nanargmax(y[stim])]
    peak_t = float(t[peak_i])
    peak = float(y[peak_i])
    dep = peak - baseline

    def first_cross(thr, start_i, end_i, direction="up"):
        if end_i <= start_i:
            return np.nan
        seg = y[start_i:end_i + 1]
        ts = t[start_i:end_i + 1]
        idx = np.where(seg >= thr)[0] if direction == "up" else np.where(seg <= thr)[0]
        return float(ts[idx[0]]) if len(idx) else np.nan

    onset_i = int(np.searchsorted(t, onset_s))
    offset_i = int(np.searchsorted(t, offset_s))
    t20 = first_cross(baseline + 0.2 * dep, onset_i, peak_i, "up")
    t63 = first_cross(baseline + 0.632 * dep, onset_i, peak_i, "up")
    t80 = first_cross(baseline + 0.8 * dep, onset_i, peak_i, "up")
    rise_slope = (0.6 * dep / (t80 - t20)) if np.isfinite(t20) and np.isfinite(t80) and t80 > t20 else np.nan
    rise_tau = t63 - onset_s if np.isfinite(t63) else np.nan

    plateau_mask = (t >= onset_s + 0.2 * (offset_s - onset_s)) & (t <= offset_s - 1.0)
    if plateau_mask.sum() > 3:
        plateau_slope = float(np.polyfit(t[plateau_mask], y[plateau_mask], 1)[0])
        plateau_level = float(np.nanmedian(y[plateau_mask]))
    else:
        plateau_slope = np.nan
        plateau_level = float(np.nanmedian(y[stim]))

    post_indices = np.where(post)[0]
    min_i = post_indices[np.nanargmin(y[post])]
    min_t = float(t[min_i])
    min_y = float(y[min_i])
    undershoot_mag = max(0.0, baseline - min_y)
    drop = plateau_level - min_y
    td80 = first_cross(plateau_level - 0.2 * drop, offset_i, min_i, "down") if drop > 0 else np.nan
    td63 = first_cross(plateau_level - 0.632 * drop, offset_i, min_i, "down") if drop > 0 else np.nan
    td20 = first_cross(plateau_level - 0.8 * drop, offset_i, min_i, "down") if drop > 0 else np.nan
    decay_slope = (0.6 * drop / (td20 - td80)) if np.isfinite(td20) and np.isfinite(td80) and td20 > td80 else np.nan
    decay_tau = td63 - offset_s if np.isfinite(td63) else np.nan
    return_slope = (float(y_raw[-1]) - min_y) / (float(t[-1]) - min_t) if float(t[-1]) > min_t else np.nan
    return {
        "baseline_mV": baseline,
        "peak_depolarization_mV": dep,
        "rise_slope_mV_per_s": rise_slope,
        "rise_tau_s": rise_tau,
        "plateau_slope_mV_per_s": plateau_slope,
        "decay_slope_mV_per_s": decay_slope,
        "decay_tau_s": decay_tau,
        "undershoot_magnitude_mV": undershoot_mag,
        "return_slope_mV_per_s": return_slope,
    }


def build_feature_pass_table(features_df: pd.DataFrame, threshold_df: Optional[pd.DataFrame], sweep_id: int, group: str = "pooled", mode: str = "min_max", min_pass_fraction: float = 0.75) -> pd.DataFrame:
    out = features_df.copy()
    if threshold_df is None or threshold_df.empty:
        out["good_enough"] = True
        out["acceptance_source"] = "top_objective_no_threshold_csv"
        out["pass_fraction"] = np.nan
        return out
    pass_cols = []
    for feature in FEATURES_FOR_FILTER:
        if feature not in out.columns:
            continue
        row = threshold_df[(threshold_df.get("group", "pooled") == group) & (threshold_df.get("sweep", sweep_id) == sweep_id) & (threshold_df.get("feature", "") == feature)]
        if row.empty:
            # Try without group filter.
            row = threshold_df[(threshold_df.get("sweep", sweep_id) == sweep_id) & (threshold_df.get("feature", "") == feature)]
        if row.empty:
            continue
        r = row.iloc[0]
        if mode == "q1_q3" and {"acceptable_low_q1", "acceptable_high_q3"}.issubset(threshold_df.columns):
            low, high = float(r["acceptable_low_q1"]), float(r["acceptable_high_q3"])
        elif mode == "ci95" and {"acceptable_low_ci95", "acceptable_high_ci95"}.issubset(threshold_df.columns):
            low, high = float(r["acceptable_low_ci95"]), float(r["acceptable_high_ci95"])
        else:
            low, high = float(r["min"]), float(r["max"])
        col = f"pass_{feature}"
        out[col] = out[feature].apply(lambda x: np.isfinite(x) and low <= x <= high)
        pass_cols.append(col)
    if not pass_cols:
        out["good_enough"] = True
        out["acceptance_source"] = "top_objective_threshold_unusable"
        out["pass_fraction"] = np.nan
    else:
        out["pass_fraction"] = out[pass_cols].sum(axis=1) / len(pass_cols)
        out["good_enough"] = out["pass_fraction"] >= min_pass_fraction
        out["acceptance_source"] = np.where(out["good_enough"], "feature_threshold", "failed_threshold")
    return out


# -----------------------------------------------------------------------------
# Mechanistic scores
# -----------------------------------------------------------------------------
def integral(t_s: np.ndarray, y: np.ndarray) -> float:
    if len(t_s) < 2:
        return np.nan
    
    try:
        return float(np.trapezoid(y, t_s))
    except AttributeError:
        return float(np.trapz(y, t_s))


def pos(x: np.ndarray) -> np.ndarray:
    return np.maximum(x, 0.0)


def neg(x: np.ndarray) -> np.ndarray:
    return np.maximum(-x, 0.0)


def first_time_fraction(t_s: np.ndarray, y: np.ndarray, window_mask: np.ndarray, frac: float = 0.1) -> float:
    if window_mask.sum() < 2:
        return np.nan
    yy = np.asarray(y, dtype=float)
    base = float(np.nanmedian(yy[~window_mask])) if (~window_mask).sum() else 0.0
    yw = np.abs(yy - base)
    peak = np.nanmax(yw[window_mask])
    if not np.isfinite(peak) or peak <= 0:
        return np.nan
    inds = np.where(window_mask & (yw >= frac * peak))[0]
    if len(inds) == 0:
        return np.nan
    return float(t_s[inds[0]])



def state_10_90(x: float, low: float = 0.10, high: float = 0.90) -> str:
    x = safe_float(x)
    if not np.isfinite(x):
        return "undefined"
    if x <= low:
        return "closed_low"
    if x >= high:
        return "open_high"
    return "partial_mid"


def classify_signed_flux_mode(D_F: float, D_I: float, dominance_margin: float = DEFAULT_DOMINANCE_MARGIN) -> str:
    """Classify a window in the corrected signed-flux/electrical-driver plane."""
    D_F = safe_float(D_F)
    D_I = safe_float(D_I)
    if not np.isfinite(D_F) or not np.isfinite(D_I):
        return "UNDEFINED"
    theta = math.log10(float(dominance_margin)) if dominance_margin and dominance_margin > 1.0 else 0.0
    flux_local = D_F > theta
    flux_spatial = D_F < -theta
    current_local = D_I > theta
    current_spatial = D_I < -theta
    if flux_local and current_local:
        return "STRICTLY_LOCAL"
    if flux_spatial and current_spatial:
        return "STRICTLY_SPATIAL"
    if flux_local and current_spatial:
        return "MIXED_LOCAL"
    if flux_spatial and current_local:
        return "MIXED_SPATIAL"
    return "BALANCED_OR_WEAK"


def linear_slope(t: np.ndarray, y: np.ndarray) -> float:
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(t) < 3 or not np.isfinite(t).all() or not np.isfinite(y).all() or np.nanmax(t) == np.nanmin(t):
        return np.nan
    try:
        return float(np.polyfit(t, y, 1)[0])
    except Exception:
        return np.nan


def trace_feature_summary(details: Dict[str, np.ndarray], context: TrialSimulationContext) -> Dict[str, object]:
    """Whole-simulation K_o and gate summaries attached to each window row."""
    t = np.asarray(details["t_s"], dtype=float)
    Ko = np.asarray(details["Ko"], dtype=float)
    gate = np.asarray(details["gate_activation"], dtype=float)
    DKa = np.asarray(details["DKa"], dtype=float)
    stim = context.windows_s.get("M_rise", (np.nan, np.nan))
    decay = context.windows_s.get("M_decay", (np.nan, np.nan))
    pre = context.windows_s.get("M0", (np.nan, np.nan))
    stim_mask = (t >= stim[0]) & (t <= stim[1])
    decay_mask = (t >= decay[0]) & (t <= decay[1])
    pre_mask = (t >= pre[0]) & (t <= pre[1])
    baseline = float(np.nanmedian(Ko[pre_mask])) if pre_mask.sum() else 4.8
    out = {"Ko_baseline_pre_mM": baseline}
    if len(Ko):
        peak_i = int(np.nanargmax(Ko)); min_i = int(np.nanargmin(Ko))
        out.update({
            "Ko_peak_global_mM": float(Ko[peak_i]),
            "Ko_peak_time_s": float(t[peak_i]),
            "Ko_min_global_mM": float(Ko[min_i]),
            "Ko_min_time_s": float(t[min_i]),
            "Ko_final_mM": float(Ko[-1]),
            "Ko_final_minus_baseline_mM": float(Ko[-1] - baseline),
            "sigmoid_gate_at_sim_end": float(gate[-1]),
            "DKa_at_sim_end": float(DKa[-1]),
        })
    if stim_mask.sum() >= 3:
        ts, Ks, gs = t[stim_mask], Ko[stim_mask], gate[stim_mask]
        peak_i = int(np.nanargmax(Ks))
        out.update({
            "Ko_amplitude_stim_mM": float(np.nanmax(Ks) - baseline),
            "Ko_stim_start_mM": float(Ks[0]),
            "Ko_stim_end_mM": float(Ks[-1]),
            "Ko_rise_rate_mM_per_s": float((Ks[peak_i] - Ks[0]) / max(ts[peak_i] - ts[0], EPS)),
            "sigmoid_gate_at_stim_start": float(gs[0]),
            "sigmoid_gate_at_stim_end": float(gs[-1]),
            "sigmoid_gate_peak_stim": float(np.nanmax(gs)),
        })
    else:
        out.update({"Ko_amplitude_stim_mM": np.nan, "Ko_rise_rate_mM_per_s": np.nan,
                    "sigmoid_gate_at_stim_start": np.nan, "sigmoid_gate_at_stim_end": np.nan, "sigmoid_gate_peak_stim": np.nan})
    if decay_mask.sum() >= 3:
        td, Kd, gd = t[decay_mask], Ko[decay_mask], gate[decay_mask]
        slope = linear_slope(td, Kd)
        out.update({
            "Ko_decay_linear_slope_mM_per_s": slope,
            "Ko_decay_rate_abs_mM_per_s": max(0.0, -slope) if np.isfinite(slope) else np.nan,
            "Ko_deep_post_mM": float(max(0.0, baseline - np.nanmin(Kd))),
            "Ko_decay_end_mM": float(Kd[-1]),
            "sigmoid_gate_at_decay_start": float(gd[0]),
            "sigmoid_gate_at_decay_end": float(gd[-1]),
            "sigmoid_gate_peak_decay": float(np.nanmax(gd)),
        })
    else:
        out.update({"Ko_decay_rate_abs_mM_per_s": np.nan, "Ko_deep_post_mM": np.nan,
                    "sigmoid_gate_at_decay_start": np.nan, "sigmoid_gate_at_decay_end": np.nan, "sigmoid_gate_peak_decay": np.nan})
    out["Ko_rise_over_decay_rate"] = out.get("Ko_rise_rate_mM_per_s", np.nan) / (out.get("Ko_decay_rate_abs_mM_per_s", np.nan) + EPS) if np.isfinite(out.get("Ko_rise_rate_mM_per_s", np.nan)) else np.nan
    out["Ko_decay_over_rise_rate"] = out.get("Ko_decay_rate_abs_mM_per_s", np.nan) / (out.get("Ko_rise_rate_mM_per_s", np.nan) + EPS) if np.isfinite(out.get("Ko_decay_rate_abs_mM_per_s", np.nan)) else np.nan
    stim_state = state_10_90(out.get("sigmoid_gate_at_stim_end", np.nan))
    end_state = state_10_90(out.get("sigmoid_gate_at_sim_end", np.nan))
    out["sigmoid_state_at_stim_end_10_90"] = stim_state
    out["sigmoid_state_at_sim_end_10_90"] = end_state
    if stim_state == "closed_low" and end_state == "open_high":
        out["temporal_recruitment_class"] = "delayed_ionic_recruitment_after_load"
    elif stim_state == "open_high" and end_state == "closed_low":
        out["temporal_recruitment_class"] = "recruited_during_load_then_closed_by_end"
    elif stim_state == "open_high" and end_state == "open_high":
        out["temporal_recruitment_class"] = "early_sustained_open_recruitment"
    elif stim_state == "closed_low" and end_state == "closed_low":
        out["temporal_recruitment_class"] = "persistently_low_range_closed"
    elif end_state == "open_high":
        out["temporal_recruitment_class"] = "fully_open_at_sim_end"
    elif end_state == "closed_low":
        out["temporal_recruitment_class"] = "fully_closed_at_sim_end"
    else:
        out["temporal_recruitment_class"] = "intermediate_or_mixed_temporal_recruitment"
    return out


def compute_window_scores(row: Mapping[str, object], details: Dict[str, np.ndarray], paramdict: Dict[str, Dict[str, object]], context: TrialSimulationContext, dominance_margin: float = DEFAULT_DOMINANCE_MARGIN) -> List[Dict[str, object]]:
    t = np.asarray(details["t_s"], dtype=float)
    A = paramdict["Astrocyte"]
    E = paramdict["external"]
    w_a = float(A["w_a"])
    w_o = float(E["w_o"])
    d_strength = float(A["d_gap"])
    pk_base = float(A["P_k"])
    Pkgap = d_strength * pk_base
    gs_value = float(A.get("gama_s", np.nan))
    gt_value = float(A.get("gama_t", np.nan))
    alpha1 = gt_value * float(A["Sig_a"]) / (float(A["w_a"]) * float(A["F"])) if np.isfinite(gt_value) else np.nan
    alpha2 = gs_value * float(A["Sig_a"]) / (float(A["w_a"]) * float(A["F"])) if np.isfinite(gs_value) else np.nan
    alpha2_over_alpha1 = alpha2 / (alpha1 + EPS) if np.isfinite(alpha1) and np.isfinite(alpha2) else np.nan
    gs_x_pkgap = gs_value * Pkgap if np.isfinite(gs_value) and np.isfinite(Pkgap) else np.nan
    gs_x_d = gs_value * d_strength if np.isfinite(gs_value) else np.nan
    trace_summary = trace_feature_summary(details, context)

    rows = []
    for window, (t1, t2) in context.windows_s.items():
        mask = (t >= t1) & (t <= t2)
        if mask.sum() < 3:
            continue
        tw = t[mask]
        dKt = details["dDKt"][mask]
        dKs = details["dKs"][mask]
        dKs_open = details["dKs_if_open"][mask]
        dKg = details["dKg"][mask]
        gate = np.clip(details["gate_activation"][mask], 0.0, 1.0)
        IKir = details["I_Kir"][mask]
        Ik = details["I_k"][mask]
        Ikgap = details["I_kgap"][mask]
        Il = details["I_l"][mask]
        Ko = details["Ko"][mask]
        Va = details["Va"][mask]
        EKa = details["EKa"][mask]
        dKo = details.get("dKo", np.gradient(details["Ko"], details["t_s"]))[mask]
        DKa = details["DKa"][mask]
        DKt_state = details["DKt"][mask]
        Ks_state = details["Ks"][mask]
        Ths = details["Ths"][mask]
        # Zhou-like r proxy: low values mean Va is closer to the syncytial clamp
        # than to the local GHK/K equilibrium. Vs is approximated by Va_s because
        # this reduced model lacks an explicit neighboring astrocyte voltage state.
        Vs_proxy = float(A.get("Va_s", -90.0))
        r_inst = np.abs((Va - Vs_proxy) / (EKa - Vs_proxy + EPS))
        r_inst = np.where(np.isfinite(r_inst), r_inst, np.nan)
        r_mean = float(np.nanmean(r_inst)) if np.isfinite(r_inst).any() else np.nan
        r_median = float(np.nanmedian(r_inst)) if np.isfinite(r_inst).any() else np.nan
        isopotentiality_from_r = 1.0 / (1.0 + r_median) if np.isfinite(r_median) else np.nan
        # K+ pump proxies. The current reduced model has no explicit Na/K ATPase state,
        # so these are post-hoc observables that can be compared with experimental
        # undershoot and recovery-phase interpretations, not literal pump fluxes.
        Ko_above_baseline = pos(Ko - 4.8)
        Ko_below_baseline = pos(4.8 - Ko)
        pump_drive_proxy = integral(tw, Ko_above_baseline)
        pump_recovery_proxy = integral(tw, pos(-dKo))
        pump_undershoot_proxy = float(np.nanmax(Ko_below_baseline)) if len(Ko_below_baseline) else np.nan
        pump_drive_x_clearance_proxy = pump_drive_proxy * pump_recovery_proxy

        # Local/long-range flux budgets.
        local_uptake = integral(tw, pos(dKt))
        local_release = integral(tw, neg(dKt))
        long_export = integral(tw, neg(dKs))
        long_import = integral(tw, pos(dKs))
        bath_source = integral(tw, pos(dKg))
        bath_sink = integral(tw, neg(dKg))
        dKs_available = integral(tw, np.abs(dKs_open))
        dKs_actual = integral(tw, np.abs(dKs))
        gate_weighted_by_available = dKs_actual / (dKs_available + EPS)
        gate_time_mean = float(np.nanmean(gate))
        gate_time_fraction_gt_05 = float(np.nanmean(gate > 0.5))
        gate_time_fraction_gt_09 = float(np.nanmean(gate > 0.9))
        gate_start = float(gate[0])
        gate_end = float(gate[-1])
        gate_peak = float(np.nanmax(gate))
        gate_end_state = state_10_90(gate_end)
        # Dynamic conversion factor proposed in the notes: alpha2 * DKa * chi_K.
        # It is a state-dependent surface/conversion term, not a literal anatomical n.
        dynamic_conversion = np.abs(DKa) * gate * (alpha2 if np.isfinite(alpha2) else np.nan)
        available_conversion_if_open = np.abs(DKa) * (alpha2 if np.isfinite(alpha2) else np.nan)
        dynamic_conversion_auc = integral(tw, dynamic_conversion)
        available_conversion_auc = integral(tw, available_conversion_if_open)
        dynamic_conversion_fraction = dynamic_conversion_auc / (available_conversion_auc + EPS)
        dynamic_conversion_peak = float(np.nanmax(dynamic_conversion)) if np.isfinite(dynamic_conversion).any() else np.nan
        # Functional n proxies: chi_K itself, current-weighted dKs activation, and Ks state.
        n_functional_time_proxy = gate_time_mean
        n_functional_flux_proxy = gate_weighted_by_available
        n_functional_end_proxy = gate_end
        # S-by-n combinations under the final interpretation: d is s_Zhou; chi/dKs is n_func.
        Sn_product_time_proxy = d_strength * n_functional_time_proxy
        Sn_product_flux_proxy = d_strength * n_functional_flux_proxy
        recruited_gap_conductance_proxy = Pkgap * n_functional_flux_proxy
        recruited_ionic_capacity_proxy = Pkgap * (alpha2 if np.isfinite(alpha2) else np.nan) * n_functional_flux_proxy

        # Updated interpretation after comparing to Cressman-style current-to-concentration
        # conversion factors: gamma_s/alpha2 is not just a numerical conversion; in a
        # reduced model it is the closest proxy for available spatial transfer area
        # divided by volume. The sigmoid/dKs activation is the recruited fraction of
        # that available surface.
        n_available_proxy_gs = gs_value
        n_available_surface_to_volume_proxy_alpha2 = alpha2
        available_surface_conductance_capacity_alpha2_Pkgap = alpha2 * Pkgap if np.isfinite(alpha2) and np.isfinite(Pkgap) else np.nan
        recruited_surface_time_proxy_gs_x_chi = gs_value * n_functional_time_proxy if np.isfinite(gs_value) else np.nan
        recruited_surface_flux_proxy_gs_x_A_dKs = gs_value * n_functional_flux_proxy if np.isfinite(gs_value) else np.nan
        recruited_surface_end_proxy_gs_x_chi_end = gs_value * n_functional_end_proxy if np.isfinite(gs_value) else np.nan
        recruited_surface_alpha2_x_A_dKs = alpha2 * n_functional_flux_proxy if np.isfinite(alpha2) else np.nan
        recruited_surface_alpha2_x_chi_end = alpha2 * n_functional_end_proxy if np.isfinite(alpha2) else np.nan
        conductance_to_surface_ratio_Pkgap_over_alpha2 = Pkgap / (alpha2 + EPS) if np.isfinite(alpha2) and np.isfinite(Pkgap) else np.nan
        surface_to_conductance_ratio_alpha2_over_Pkgap = alpha2 / (Pkgap + EPS) if np.isfinite(alpha2) and np.isfinite(Pkgap) else np.nan

        # Current budgets.
        I_local_abs = integral(tw, np.abs(IKir) + np.abs(Ik))
        I_kir_abs = integral(tw, np.abs(IKir))
        I_gap_abs = integral(tw, np.abs(Ikgap))
        I_leak_abs = integral(tw, np.abs(Il))
        I_total_abs = I_local_abs + I_gap_abs + I_leak_abs + EPS
        voltage_coupling_score = I_gap_abs / I_total_abs
        kir_current_score = I_kir_abs / I_total_abs
        leak_score = I_leak_abs / I_total_abs

        # Signed current-direction budget retained from the flux-budget classifier.
        # Positive currents are outward in this model convention; negative currents are inward.
        current_stack = np.vstack([IKir, Ik, Ikgap, Il])
        inward_time = np.sum(np.where(current_stack < 0.0, np.abs(current_stack), 0.0), axis=0)
        outward_time = np.sum(np.where(current_stack > 0.0, np.abs(current_stack), 0.0), axis=0)
        I_inward_abs_integral = integral(tw, inward_time)
        I_outward_abs_integral = integral(tw, outward_time)
        I_ss = I_inward_abs_integral - I_outward_abs_integral
        I_tot_directional = I_inward_abs_integral + I_outward_abs_integral + EPS
        local_current_abs_IKir_plus_Ik = integral(tw, np.abs(IKir + Ik))

        # Core scores.
        local_amt = w_a * local_uptake
        long_amt = w_a * long_export
        source_amt = w_o * bath_source
        sink_amt = w_o * bath_sink
        low_long_fraction = long_amt / (local_amt + long_amt + EPS)
        local_fraction = local_amt / (local_amt + long_amt + EPS)
        flux_log10_local_over_long = math.log10((local_amt + EPS) / (long_amt + EPS))
        current_log10_kir_over_gap = math.log10((I_kir_abs + EPS) / (I_gap_abs + EPS))
        D_F_log10_local_load_over_spatial_export = math.log10((local_uptake + EPS) / (long_export + EPS))
        D_I_elec_log10_local_current_over_gap_current = math.log10((local_current_abs_IKir_plus_Ik + EPS) / (I_gap_abs + EPS))
        source_balance_index = (bath_source - bath_sink) / (bath_source + bath_sink + EPS)
        spatial_directionality_index = (long_export - long_import) / (long_export + long_import + EPS)
        local_directionality_index = (local_uptake - local_release) / (local_uptake + local_release + EPS)
        signed_flux_mode = classify_signed_flux_mode(D_F_log10_local_load_over_spatial_export, D_I_elec_log10_local_current_over_gap_current, dominance_margin)
        mode_confidence_log10 = min(abs(D_F_log10_local_load_over_spatial_export), abs(D_I_elec_log10_local_current_over_gap_current))
        ionic_coupling_score = gate_weighted_by_available * voltage_coupling_score
        source_clearance_ratio = (local_amt + long_amt) / (source_amt + EPS)
        buffering_efficiency = 1.0 - (integral(tw, pos(Ko - 4.8)) / ((source_amt / max(w_o, EPS)) + EPS))

        # Phenotype-friendly latencies in stimulation window, useful for "voltage first/ionic catch-up".
        t10_gate = first_time_fraction(t, details["gate_activation"], mask, 0.1)
        t90_gate = first_time_fraction(t, details["gate_activation"], mask, 0.9)
        t10_dKs = first_time_fraction(t, np.abs(details["dKs"]), mask, 0.1)
        t10_Igap = first_time_fraction(t, np.abs(details["I_kgap"]), mask, 0.1)
        lag_dKs_vs_Igap = t10_dKs - t10_Igap if np.isfinite(t10_dKs) and np.isfinite(t10_Igap) else np.nan

        if gate_weighted_by_available < 0.10:
            gj_state = "closed_low_redistribution"
        elif gate_weighted_by_available > 0.90:
            gj_state = "open_long_range_redistribution"
        else:
            gj_state = "intermediate_recruitment"

        rows.append({
            "trial_number": int(row["trial_number"]),
            "trial_id": int(row["trial_id"]),
            "db_file": row.get("db_file", ""),
            "experiment": context.experiment_type,
            "current_na": context.current_na,
            "window": window,
            "objective": safe_float(row.get("objective")),
            "gki": safe_float(row.get("gki")),
            "d": safe_float(row.get("d")),
            "pk": safe_float(row.get("pk")),
            "Pkgap": Pkgap,
            "zhou_s_coupling_strength_proxy_d": d_strength,
            "pk_base_permeability_proxy": pk_base,
            "GJ_conductance_proxy_Pkgap": Pkgap,
            "gs": gs_value,
            "gt": gt_value,
            "alpha1_transmembrane_current_to_flux": alpha1,
            "alpha2_intercellular_current_to_flux": alpha2,
            "alpha2_over_alpha1": alpha2_over_alpha1,
            "gs_x_Pkgap": gs_x_pkgap,
            "gs_x_d": gs_x_d,
            "n_available_proxy_gs": n_available_proxy_gs,
            "n_available_surface_to_volume_proxy_alpha2": n_available_surface_to_volume_proxy_alpha2,
            "available_surface_conductance_capacity_alpha2_Pkgap": available_surface_conductance_capacity_alpha2_Pkgap,
            "recruited_surface_time_proxy_gs_x_chi": recruited_surface_time_proxy_gs_x_chi,
            "recruited_surface_flux_proxy_gs_x_A_dKs": recruited_surface_flux_proxy_gs_x_A_dKs,
            "recruited_surface_end_proxy_gs_x_chi_end": recruited_surface_end_proxy_gs_x_chi_end,
            "recruited_surface_alpha2_x_A_dKs": recruited_surface_alpha2_x_A_dKs,
            "recruited_surface_alpha2_x_chi_end": recruited_surface_alpha2_x_chi_end,
            "conductance_to_surface_ratio_Pkgap_over_alpha2": conductance_to_surface_ratio_Pkgap_over_alpha2,
            "surface_to_conductance_ratio_alpha2_over_Pkgap": surface_to_conductance_ratio_alpha2_over_Pkgap,
            "eps": safe_float(row.get("eps")),
            "eps_middle": safe_float(row.get("eps_middle", 1.0)),
            "K_bath_value_middle": safe_float(row.get("K_bath_value_middle")),
            "drive_eps_x_deltaKbath": safe_float(row.get("drive_eps_x_deltaKbath")),
            "drive_eps_x_Kbath": safe_float(row.get("drive_eps_x_Kbath")),
            "wa_over_wo": safe_float(row.get("wa_over_wo")),
            "switching_function": row.get("switching_function", ""),
            "zth": safe_float(row.get("zth")),
            "zs": safe_float(row.get("zs")),
            "local_uptake_amount": local_amt,
            "local_release_amount": w_a * local_release,
            "long_range_export_amount": long_amt,
            "long_range_import_amount": w_a * long_import,
            "Ks_peak": float(np.nanmax(Ks_state)),
            "Ks_min": float(np.nanmin(Ks_state)),
            "Ks_final": float(Ks_state[-1]),
            "Ks_abs_auc": integral(tw, np.abs(Ks_state)),
            "DKt_peak": float(np.nanmax(DKt_state)),
            "DKt_final": float(DKt_state[-1]),
            "DKt_abs_auc": integral(tw, np.abs(DKt_state)),
            "Ths_abs_auc": integral(tw, np.abs(Ths)),
            "bath_source_amount": source_amt,
            "bath_sink_amount": sink_amt,
            "dKs_actual_abs_integral": dKs_actual,
            "dKs_available_if_open_abs_integral": dKs_available,
            "dKs_activation_score": gate_weighted_by_available,
            "sigmoid_activation_mean": gate_time_mean,
            "sigmoid_fraction_gt_0p5": gate_time_fraction_gt_05,
            "sigmoid_fraction_gt_0p9": gate_time_fraction_gt_09,
            "sigmoid_gate_start_value": gate_start,
            "sigmoid_gate_end_value": gate_end,
            "sigmoid_gate_peak_value": gate_peak,
            "sigmoid_gate_end_state_10_90": gate_end_state,
            "dynamic_conversion_factor_auc": dynamic_conversion_auc,
            "dynamic_conversion_factor_if_open_auc": available_conversion_auc,
            "dynamic_conversion_factor_fraction": dynamic_conversion_fraction,
            "dynamic_conversion_factor_peak": dynamic_conversion_peak,
            "functional_n_time_proxy_chi_mean": n_functional_time_proxy,
            "functional_n_flux_proxy_dKs_activation": n_functional_flux_proxy,
            "functional_n_end_proxy_chi_end": n_functional_end_proxy,
            "S_times_n_time_proxy_d_x_chi": Sn_product_time_proxy,
            "S_times_n_flux_proxy_d_x_A_dKs": Sn_product_flux_proxy,
            "recruited_gap_conductance_proxy_Pkgap_x_nfunc": recruited_gap_conductance_proxy,
            "recruited_ionic_capacity_proxy_Pkgap_alpha2_nfunc": recruited_ionic_capacity_proxy,
            "r_ionic_contribution_mean": r_mean,
            "r_ionic_contribution_median": r_median,
            "isopotentiality_score_from_r": isopotentiality_from_r,
            "gj_ionic_state_10_90": gj_state,
            "low_range_local_fraction": local_fraction,
            "long_range_distribution_fraction": low_long_fraction,
            "flux_log10_local_over_long": flux_log10_local_over_long,
            "I_kir_abs": I_kir_abs,
            "I_gap_abs": I_gap_abs,
            "I_leak_abs": I_leak_abs,
            "I_total_abs": I_total_abs,
            "local_current_abs_IKir_plus_Ik": local_current_abs_IKir_plus_Ik,
            "I_inward_abs_integral": I_inward_abs_integral,
            "I_outward_abs_integral": I_outward_abs_integral,
            "I_ss_inward_minus_outward": I_ss,
            "I_tot_inward_plus_outward": I_tot_directional,
            "D_F_log10_local_load_over_spatial_export": D_F_log10_local_load_over_spatial_export,
            "D_I_elec_log10_local_current_over_gap_current": D_I_elec_log10_local_current_over_gap_current,
            "mechanistic_mode_signed_flux": signed_flux_mode,
            "mode_confidence_log10": mode_confidence_log10,
            "dominance_margin_for_mode": dominance_margin,
            "source_balance_index": source_balance_index,
            "spatial_directionality_index": spatial_directionality_index,
            "local_directionality_index": local_directionality_index,
            "voltage_coupling_score": voltage_coupling_score,
            "kir_current_score": kir_current_score,
            "leak_current_score": leak_score,
            "ionic_coupling_score": ionic_coupling_score,
            "current_log10_kir_over_gap": current_log10_kir_over_gap,
            "source_clearance_ratio": source_clearance_ratio,
            "buffering_efficiency_index": buffering_efficiency,
            "pump_drive_proxy_Ko_AUC": pump_drive_proxy,
            "pump_recovery_proxy_Ko_clearance": pump_recovery_proxy,
            "pump_undershoot_proxy_mM": pump_undershoot_proxy,
            "pump_drive_x_clearance_proxy": pump_drive_x_clearance_proxy,
            "Ko_min": float(np.nanmin(Ko)),
            "Ko_peak": float(np.nanmax(Ko)),
            "Ko_auc_above_baseline": integral(tw, pos(Ko - 4.8)),
            **trace_summary,
            "Va_min": float(np.nanmin(Va)),
            "Va_max": float(np.nanmax(Va)),
            "EKa_min": float(np.nanmin(EKa)),
            "EKa_max": float(np.nanmax(EKa)),
            "mean_Va_minus_EK": float(np.nanmean(Va - EKa)),
            "t10_sigmoid_activation_s": t10_gate,
            "t90_sigmoid_activation_s": t90_gate,
            "t10_dKs_abs_s": t10_dKs,
            "t10_Igap_abs_s": t10_Igap,
            "lag_dKs_vs_Igap_s": lag_dKs_vs_Igap,
        })
    return rows


# -----------------------------------------------------------------------------
# Phenotype classification
# -----------------------------------------------------------------------------
def add_quantile_bins(df: pd.DataFrame, scope_cols: Sequence[str], low_q: float = 0.33, high_q: float = 0.67) -> Tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    specs = {
        "gki": "kir_bin",
        "d": "zhou_s_bin",
        "Pkgap": "pkgap_bin",
        "GJ_conductance_proxy_Pkgap": "gj_conductance_bin",
        "pk": "pk_bin",
        "gs": "gs_bin",
        "alpha2_intercellular_current_to_flux": "alpha2_bin",
        "alpha2_over_alpha1": "alpha2_alpha1_bin",
        "gs_x_Pkgap": "gs_x_pkgap_bin",
        "n_available_proxy_gs": "n_available_bin",
        "n_available_surface_to_volume_proxy_alpha2": "available_surface_bin",
        "available_surface_conductance_capacity_alpha2_Pkgap": "available_surface_conductance_bin",
        "recruited_surface_flux_proxy_gs_x_A_dKs": "recruited_surface_bin",
        "recruited_surface_alpha2_x_A_dKs": "recruited_alpha2_surface_bin",
        "conductance_to_surface_ratio_Pkgap_over_alpha2": "conductance_surface_ratio_bin",
        "surface_to_conductance_ratio_alpha2_over_Pkgap": "surface_conductance_ratio_bin",
        "dynamic_conversion_factor_fraction": "dynamic_conversion_bin",
        "functional_n_flux_proxy_dKs_activation": "functional_n_bin",
        "functional_n_end_proxy_chi_end": "functional_n_end_bin",
        "S_times_n_flux_proxy_d_x_A_dKs": "S_times_n_bin",
        "recruited_ionic_capacity_proxy_Pkgap_alpha2_nfunc": "recruited_capacity_bin",
        "r_ionic_contribution_median": "r_bin",
        "isopotentiality_score_from_r": "isopotentiality_bin",
        "Ks_abs_auc": "Ks_auc_bin",
        "DKt_abs_auc": "DKt_auc_bin",
        "drive_eps_x_deltaKbath": "drive_bin",
        "voltage_coupling_score": "voltage_coupling_bin",
        "dKs_activation_score": "activation_bin",
        "long_range_distribution_fraction": "long_range_bin",
        "Ko_amplitude_stim_mM": "Ko_amp_bin",
        "Ko_deep_post_mM": "Ko_deep_bin",
        "Ko_rise_over_decay_rate": "rise_decay_bin",
        "pump_drive_proxy_Ko_AUC": "pump_drive_bin",
        "pump_undershoot_proxy_mM": "pump_undershoot_bin",
    }
    rows = []
    groups = [((), out)] if not scope_cols else list(out.groupby(list(scope_cols), dropna=False))
    for group_key, g in groups:
        idx = g.index
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        group_info = {scope_cols[i]: group_key[i] for i in range(len(scope_cols))} if scope_cols else {}
        for col, bin_col in specs.items():
            if col not in out.columns:
                continue
            vals = pd.to_numeric(g[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            if len(vals) < 3:
                lo, hi = np.nan, np.nan
            else:
                lo, hi = float(vals.quantile(low_q)), float(vals.quantile(high_q))
            rows.append({**group_info, "score": col, "low_threshold": lo, "high_threshold": hi, "low_q": low_q, "high_q": high_q})
            def label(x):
                x = safe_float(x)
                if not np.isfinite(x) or not np.isfinite(lo) or not np.isfinite(hi):
                    return "unknown"
                if x <= lo:
                    return "low"
                if x >= hi:
                    return "high"
                return "mid"
            out.loc[idx, bin_col] = g[col].apply(label)
    return out, pd.DataFrame(rows)


def classify_phenotype(row: Mapping[str, object], mfa_remaining_fraction: float = 0.286) -> Dict[str, object]:
    """Final biophysical classifier.

    Mapping used:
      d                  -> Zhou s-like per-link/tightness proxy
      Pkgap=pk*d         -> total GJ conductance/permeability proxy
      gs/alpha2           -> available anatomical/surface-to-volume syncytium-capacity proxy
      chi_K/A_dKs/Ks      -> recruited fraction/state of that available syncytium
      alpha2*A_dKs        -> recruited surface capacity; spatial current-to-flux gain
      r_model            -> ionic contribution versus electrical clamp tag

    The label keeps d/S and Pkgap/GJ separate. A configuration may have low d but
    high Pkgap if pk is high; this is tagged as high total GJ conductance, not as
    high Zhou-S tightness.
    """
    kir = row.get("kir_bin", "unknown")
    s_bin = row.get("zhou_s_bin", "unknown")
    gj_bin = row.get("gj_conductance_bin", row.get("pkgap_bin", "unknown"))
    n_bin = row.get("functional_n_bin", row.get("activation_bin", "unknown"))
    n_end_bin = row.get("functional_n_end_bin", "unknown")
    dynamic_bin = row.get("dynamic_conversion_bin", "unknown")
    alpha2_bin = row.get("alpha2_bin", "unknown")
    n_available_bin = row.get("n_available_bin", row.get("gs_bin", "unknown"))
    available_surface_bin = row.get("available_surface_bin", alpha2_bin)
    recruited_surface_bin = row.get("recruited_surface_bin", "unknown")
    surface_conductance_bin = row.get("surface_conductance_ratio_bin", "unknown")
    conductance_surface_bin = row.get("conductance_surface_ratio_bin", "unknown")
    r_bin = row.get("r_bin", "unknown")
    iso_bin = row.get("isopotentiality_bin", "unknown")
    Ks_bin = row.get("Ks_auc_bin", "unknown")
    DKt_bin = row.get("DKt_auc_bin", "unknown")
    voltage = row.get("voltage_coupling_bin", "unknown")
    pump_bin = row.get("pump_drive_bin", "unknown")
    pump_undershoot_bin = row.get("pump_undershoot_bin", "unknown")
    temporal = str(row.get("temporal_recruitment_class", ""))
    gj_state = row.get("gj_ionic_state_10_90", "")

    s_high = s_bin == "high"
    s_low = s_bin == "low"
    gj_high = gj_bin == "high"
    gj_low = gj_bin == "low"
    # Separate anatomical availability from recruited state. This is the key gs correction:
    # gs/alpha2 is available surface/volume capacity; A_dKs/chi is recruited fraction.
    big_available_n = n_available_bin == "high" or available_surface_bin == "high"
    small_available_n = n_available_bin == "low" and available_surface_bin == "low"
    big_recruited_n = n_bin == "high" or recruited_surface_bin == "high" or n_end_bin == "high" or gj_state == "open_long_range_redistribution"
    small_recruited_n = n_bin == "low" or gj_state == "closed_low_redistribution"
    # Keep the historical names as aliases for downstream rules.
    big_n = big_recruited_n
    small_n = small_recruited_n
    high_alpha = alpha2_bin == "high" or dynamic_bin == "high" or recruited_surface_bin == "high"
    low_r_or_high_iso = (r_bin == "low") or (iso_bin == "high")
    high_Ks = Ks_bin == "high"
    high_DKt = DKt_bin == "high"

    if big_available_n and small_recruited_n and gj_high:
        pheno = "largeAvailableSurface_highGJ_but_unrecruited"
        rationale = "gs/alpha2 indicates large available anatomical/surface capacity and Pkgap/GJ is high, but dKs/chi recruitment remains low; available syncytium is present but functionally silent for ionic redistribution"
    elif small_available_n and gj_high and voltage == "high":
        pheno = "smallAvailableSurface_highGJ_voltageClamp_surfaceLimited"
        rationale = "high GJ conductance/current provides electrical coupling, but gs/alpha2 surface capacity is low; spatial ionic flux is surface/volume limited"
    elif big_available_n and gj_low and big_recruited_n:
        pheno = "largeAvailableSurface_lowGJ_recruitedButWeakCoupled"
        rationale = "gs/alpha2 and recruitment are high but Pkgap/GJ is low; broad available surface can compensate partially for weak coupling strength"
    elif big_available_n and big_recruited_n and gj_high and low_r_or_high_iso:
        pheno = "largeAvailableAndRecruitedSurface_highGJ_strongIsopotential"
        rationale = "available surface capacity, recruited surface, and GJ conductance are all high; low-r/high-isopotentiality supports strong syncytial voltage clamp and ionic redistribution"
    elif "delayed_ionic_recruitment" in temporal:
        pheno = "delayed_surface_recruitment_local_storage_then_spatial_catchup"
        rationale = "sigmoid/dynamic conversion stays low during load but is open at simulation end; local K burden accumulates before spatial flux catches up"
    elif big_n and s_high and gj_high and low_r_or_high_iso:
        pheno = "bigN_highS_highGJ_strong_isopotential_long_range"
        rationale = "high functional recruitment, high d/S tightness, high Pkgap/GJ conductance, and low-r/high-isopotentiality"
    elif big_n and s_low and gj_high:
        pheno = "bigN_lowS_but_highGJ_permeability_compensated"
        rationale = "large functional recruitment with low d/S but high total Pkgap, likely because pk compensates; wide range with strong conductance proxy"
    elif small_n and gj_high and voltage == "high":
        pheno = "smallN_highGJ_voltageCoupled_local_range"
        rationale = "high GJ conductance/current with small functional recruited syncytium; electrical benefit without broad ionic redistribution"
    elif big_n and gj_low:
        pheno = "bigN_lowGJ_large_range_weak_coupling"
        rationale = "large functional recruitment but weak total gap conductance; size can compensate only partially"
    elif small_n and s_low and gj_low:
        pheno = "smallN_lowS_lowGJ_GHK_like_local"
        rationale = "low coupling tightness, low total conductance, and low functional recruitment; closest to local GHK/Kir behavior"
    elif voltage == "high" and small_n:
        pheno = "voltage_coupled_ionic_closed"
        rationale = "gap current contributes to voltage before meaningful dKs/spatial flux recruitment"
    elif high_Ks and small_n and high_alpha:
        pheno = "closed_or_smallN_high_alpha2_tight_flux_conversion"
        rationale = "high Ks/spatial state despite low activation; alpha2/dynamic conversion suggests tight local conversion rather than broad range"
    elif high_Ks and big_n and low_r_or_high_iso and high_alpha:
        pheno = "large_functional_syncytium_highKs_lowR_highAlpha"
        rationale = "high Ks, large functional recruitment, low-r/high-isopotentiality, and high alpha2"
    elif kir == "high" and small_n:
        pheno = "DH_like_highKir_smallFunctionalN"
        rationale = "high Kir with low functional syncytial recruitment; local membrane entry/control phenotype"
    elif kir == "low" and big_n and (pump_bin == "high" or pump_undershoot_bin == "high"):
        pheno = "VH_like_lowKir_bigFunctionalN_pumpCompensated"
        rationale = "low Kir with large functional recruitment and strong K recovery/undershoot proxy"
    elif high_DKt and small_n:
        pheno = "local_Kt_retention_low_range"
        rationale = "large local Kt burden with low spatial recruitment"
    else:
        pheno = "intermediate_or_degenerate_final_axes"
        rationale = "mixed n/S/GJ/r/alpha/Ks pattern; retain quantitative tags for data-driven refinement"

    predicted_mfa_sensitivity = safe_float(row.get("voltage_coupling_score", np.nan)) * (1.0 - mfa_remaining_fraction)
    predicted_ba_sensitivity = safe_float(row.get("kir_current_score", np.nan))
    predicted_pump_activity = safe_float(row.get("pump_drive_proxy_Ko_AUC", np.nan)) * safe_float(row.get("pump_recovery_proxy_Ko_clearance", np.nan))
    return {
        "buffering_phenotype": pheno,
        "phenotype_rationale": rationale,
        "final_n_by_S_tag": f"nAvail_{n_available_bin}__nRecruit_{n_bin}__nEnd_{n_end_bin}__S_d_{s_bin}__GJ_{gj_bin}",
        "final_surface_conductance_tag": f"surface_{available_surface_bin}__recruitedSurface_{recruited_surface_bin}__condOverSurf_{conductance_surface_bin}__surfOverCond_{surface_conductance_bin}",
        "final_r_alpha_K_tag": f"r_{r_bin}__iso_{iso_bin}__alpha_{alpha2_bin}__Ks_{Ks_bin}",
        "predicted_MFA_sensitivity_score": predicted_mfa_sensitivity,
        "predicted_Ba_sensitivity_score": predicted_ba_sensitivity,
        "predicted_pump_activity_proxy": predicted_pump_activity,
    }

def add_phenotypes(scores_df: pd.DataFrame, bin_scope: str, low_q: float, high_q: float, mfa_remaining_fraction: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    scope_cols: List[str]
    if bin_scope == "experiment_current_window":
        scope_cols = ["experiment", "current_na", "window"]
    elif bin_scope == "experiment_window":
        scope_cols = ["experiment", "window"]
    elif bin_scope == "window":
        scope_cols = ["window"]
    else:
        scope_cols = []
    binned, thresholds = add_quantile_bins(scores_df, scope_cols, low_q=low_q, high_q=high_q)
    pheno_rows = [classify_phenotype(r, mfa_remaining_fraction) for _, r in binned.iterrows()]
    pheno_df = pd.DataFrame(pheno_rows, index=binned.index)
    return pd.concat([binned, pheno_df], axis=1), thresholds


# -----------------------------------------------------------------------------
# Plotting
# -----------------------------------------------------------------------------
def plot_mode_space(df: pd.DataFrame, out_dir: Path) -> None:
    plot_dir = out_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    for window, g in df.groupby("window"):
        g = g.replace([np.inf, -np.inf], np.nan).dropna(subset=["dKs_activation_score", "long_range_distribution_fraction", "kir_current_score", "voltage_coupling_score"])
        if g.empty:
            continue
        plt.figure(figsize=(7.2, 5.8))
        for pheno, gp in g.groupby("buffering_phenotype"):
            plt.scatter(gp["dKs_activation_score"], gp["kir_current_score"] - gp["voltage_coupling_score"], s=18, alpha=0.65, label=pheno[:38])
        plt.axvline(0.10, linestyle="--", linewidth=1)
        plt.axvline(0.90, linestyle="--", linewidth=1)
        plt.axhline(0.0, linestyle="--", linewidth=1)
        plt.xlabel("dKs activation score: actual / fully-open spatial flux")
        plt.ylabel("Kir current score - gap voltage-coupling score")
        plt.title(f"Phenotype mode space — {window}")
        plt.legend(fontsize=7, loc="best")
        plt.tight_layout()
        plt.savefig(plot_dir / f"phenotype_mode_space_{window}.png", dpi=180)
        plt.close()


def plot_activation_histograms(df: pd.DataFrame, out_dir: Path) -> None:
    plot_dir = out_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    for window, g in df.groupby("window"):
        plt.figure(figsize=(7.0, 4.6))
        plt.hist(g["dKs_activation_score"].replace([np.inf, -np.inf], np.nan).dropna(), bins=30, alpha=0.75)
        plt.axvline(0.10, linestyle="--", linewidth=1)
        plt.axvline(0.90, linestyle="--", linewidth=1)
        plt.xlabel("dKs activation score")
        plt.ylabel("Configurations")
        plt.title(f"Open/intermediate/closed dKs recruitment — {window}")
        plt.tight_layout()
        plt.savefig(plot_dir / f"dKs_activation_histogram_{window}.png", dpi=180)
        plt.close()


def plot_parameter_projection(df: pd.DataFrame, out_dir: Path) -> None:
    if not SKLEARN_AVAILABLE:
        return
    plot_dir = out_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    cols = ["gki", "Pkgap", "d", "pk", "eps", "K_bath_value_middle", "drive_eps_x_deltaKbath", "wa_over_wo", "zth", "zs"]
    use = [c for c in cols if c in df.columns]
    g = df.drop_duplicates(["experiment", "current_na", "trial_number"])[["experiment", "current_na", "trial_number", "buffering_phenotype"] + use].copy()
    if len(g) < 3 or not use:
        return
    X = g[use].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    Xs = StandardScaler().fit_transform(X)
    pcs = PCA(n_components=2, random_state=0).fit_transform(Xs)
    g["PC1"] = pcs[:, 0]
    g["PC2"] = pcs[:, 1]
    plt.figure(figsize=(7.2, 5.8))
    for pheno, gp in g.groupby("buffering_phenotype"):
        plt.scatter(gp["PC1"], gp["PC2"], s=28, alpha=0.75, label=pheno[:38])
    plt.xlabel("PC1 of parameter vector")
    plt.ylabel("PC2")
    plt.title("Accepted configurations projected by predicted phenotype")
    plt.legend(fontsize=7, loc="best")
    plt.tight_layout()
    plt.savefig(plot_dir / "parameter_projection_by_phenotype.png", dpi=180)
    plt.close()
    g.to_csv(out_dir / "tables" / "parameter_projection_by_phenotype.csv", index=False)



def add_mfa_control_contrasts(phenotyped: pd.DataFrame, table_dir: Path) -> None:
    """Save MFA-vs-CONTROL contrasts by current/window for phenotype matching.

    These contrasts are not paired by trial identity. They compare the distributions of
    accepted model configurations for CONTROL versus MFA at the same current and window.
    """
    if phenotyped.empty or not {"CONTROL", "MFA"}.issubset(set(phenotyped["experiment"].astype(str).unique())):
        return
    metrics = [
        "dKs_activation_score", "sigmoid_activation_mean", "long_range_distribution_fraction",
        "voltage_coupling_score", "kir_current_score", "ionic_coupling_score", "gki", "Pkgap", "d",
        "gs", "gt", "alpha1_transmembrane_current_to_flux", "alpha2_intercellular_current_to_flux", "alpha2_over_alpha1",
        "gs_x_Pkgap", "gs_x_d", "n_available_proxy_gs", "n_available_surface_to_volume_proxy_alpha2",
        "available_surface_conductance_capacity_alpha2_Pkgap", "recruited_surface_flux_proxy_gs_x_A_dKs",
        "recruited_surface_alpha2_x_A_dKs", "conductance_to_surface_ratio_Pkgap_over_alpha2", "surface_to_conductance_ratio_alpha2_over_Pkgap",
        "dynamic_conversion_factor_fraction", "dynamic_conversion_factor_peak",
        "functional_n_flux_proxy_dKs_activation", "functional_n_end_proxy_chi_end",
        "S_times_n_flux_proxy_d_x_A_dKs", "recruited_ionic_capacity_proxy_Pkgap_alpha2_nfunc",
        "r_ionic_contribution_median", "isopotentiality_score_from_r", "Ks_abs_auc", "DKt_abs_auc",
        "Ko_peak", "Ko_auc_above_baseline", "Ko_amplitude_stim_mM", "Ko_deep_post_mM", "Ko_rise_over_decay_rate",
        "pump_drive_proxy_Ko_AUC", "pump_recovery_proxy_Ko_clearance", "pump_undershoot_proxy_mM",
        "predicted_MFA_sensitivity_score", "predicted_Ba_sensitivity_score", "predicted_pump_activity_proxy",
    ]
    available = [m for m in metrics if m in phenotyped.columns]
    rows = []
    for (current, window), g in phenotyped.groupby(["current_na", "window"], dropna=False):
        c = g[g["experiment"] == "CONTROL"]
        m = g[g["experiment"] == "MFA"]
        if c.empty or m.empty:
            continue
        base = {"current_na": current, "window": window, "n_CONTROL": int(len(c)), "n_MFA": int(len(m))}
        for col in available:
            cv = pd.to_numeric(c[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            mv = pd.to_numeric(m[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            if len(cv) == 0 or len(mv) == 0:
                continue
            cmean, mmean = float(cv.mean()), float(mv.mean())
            base[f"CONTROL_mean_{col}"] = cmean
            base[f"MFA_mean_{col}"] = mmean
            base[f"MFA_minus_CONTROL_{col}"] = mmean - cmean
            base[f"MFA_over_CONTROL_{col}"] = mmean / (cmean + EPS)
        rows.append(base)
    contrast = pd.DataFrame(rows)
    contrast.to_csv(table_dir / "mfa_control_contrast_by_current_window.csv", index=False)

    # Phenotype frequency shifts.
    counts = phenotyped.groupby(["experiment", "current_na", "window", "buffering_phenotype"]).size().reset_index(name="n")
    if counts.empty:
        return
    total = counts.groupby(["experiment", "current_na", "window"])["n"].transform("sum")
    counts["fraction"] = counts["n"] / total
    ctrl = counts[counts["experiment"] == "CONTROL"].rename(columns={"n":"n_CONTROL", "fraction":"fraction_CONTROL"})
    mfa = counts[counts["experiment"] == "MFA"].rename(columns={"n":"n_MFA", "fraction":"fraction_MFA"})
    shift = pd.merge(
        ctrl[["current_na", "window", "buffering_phenotype", "n_CONTROL", "fraction_CONTROL"]],
        mfa[["current_na", "window", "buffering_phenotype", "n_MFA", "fraction_MFA"]],
        on=["current_na", "window", "buffering_phenotype"], how="outer"
    ).fillna(0)
    shift["fraction_MFA_minus_CONTROL"] = shift["fraction_MFA"] - shift["fraction_CONTROL"]
    shift.to_csv(table_dir / "mfa_control_phenotype_shift_by_current_window.csv", index=False)

    # GS range and accepted parameter ranges.
    range_cols = [c for c in ["gki", "Pkgap", "d", "pk", "gs", "gt", "alpha2_intercellular_current_to_flux", "alpha2_over_alpha1", "gs_x_Pkgap", "gs_x_d", "n_available_proxy_gs", "n_available_surface_to_volume_proxy_alpha2", "available_surface_conductance_capacity_alpha2_Pkgap", "recruited_surface_flux_proxy_gs_x_A_dKs", "recruited_surface_alpha2_x_A_dKs", "conductance_to_surface_ratio_Pkgap_over_alpha2", "surface_to_conductance_ratio_alpha2_over_Pkgap", "pump_drive_proxy_Ko_AUC", "pump_undershoot_proxy_mM"] if c in phenotyped.columns]
    range_rows = []
    for (experiment, current, window), g in phenotyped.groupby(["experiment", "current_na", "window"], dropna=False):
        for col in range_cols:
            vals = pd.to_numeric(g[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            if vals.empty:
                continue
            range_rows.append({
                "experiment": experiment, "current_na": current, "window": window, "parameter_or_score": col,
                "n": int(len(vals)), "min": float(vals.min()), "q1": float(vals.quantile(0.25)),
                "median": float(vals.median()), "q3": float(vals.quantile(0.75)), "max": float(vals.max()),
                "mean": float(vals.mean()), "std": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
            })
    pd.DataFrame(range_rows).to_csv(table_dir / "gs_pkgap_pump_ranges_by_experiment_current_window.csv", index=False)

# -----------------------------------------------------------------------------
# Main workflow
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# Unified notebook outputs: F dictionaries, mode vectors, ranges, hidden overlays
# -----------------------------------------------------------------------------
def numeric_summary_dict(values: pd.Series) -> Dict[str, float]:
    x = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if x.empty:
        return {"count": 0, "min": np.nan, "q25": np.nan, "median": np.nan, "q75": np.nan, "max": np.nan, "range": np.nan, "mean": np.nan, "std": np.nan}
    return {
        "count": int(x.shape[0]),
        "min": float(x.min()),
        "q25": float(x.quantile(0.25)),
        "median": float(x.median()),
        "q75": float(x.quantile(0.75)),
        "max": float(x.max()),
        "range": float(x.max() - x.min()),
        "mean": float(x.mean()),
        "std": float(x.std(ddof=1)) if x.shape[0] > 1 else 0.0,
    }


def build_feature_variability_outputs(features: pd.DataFrame, table_dir: Path) -> Tuple[Dict[str, Dict[str, float]], Dict[str, Dict[str, float]], pd.DataFrame]:
    accepted = features[features.get("good_enough", False).astype(bool)].copy() if "good_enough" in features.columns else features.copy()
    rows = []
    dictionaries: Dict[str, Dict[str, Dict[str, float]]] = {}
    for prefix, name in [("V_", "Fv"), ("Ko_", "Fk")]:
        feature_cols = [c for c in accepted.columns if c.startswith(prefix)]
        dct: Dict[str, Dict[str, float]] = {}
        for col in feature_cols:
            bare = col[len(prefix):]
            stats_d = numeric_summary_dict(accepted[col])
            dct[bare] = stats_d
            rows.append({"dictionary": name, "feature": bare, **stats_d})
        dictionaries[name] = dct
    Fv = dictionaries.get("Fv", {})
    Fk = dictionaries.get("Fk", {})
    table = pd.DataFrame(rows)
    with open(table_dir / "Fv_membrane_feature_variability.json", "w") as f:
        json.dump(Fv, f, indent=2, allow_nan=True)
    with open(table_dir / "Fk_Ko_feature_variability.json", "w") as f:
        json.dump(Fk, f, indent=2, allow_nan=True)
    table.to_csv(table_dir / "feature_variability_Fv_Fk.csv", index=False)
    return Fv, Fk, table


def build_mode_vector_outputs(phenotyped: pd.DataFrame, table_dir: Path) -> Tuple[pd.DataFrame, Dict[str, Dict[str, object]]]:
    if phenotyped.empty:
        return pd.DataFrame(), {}
    idx_cols = ["experiment", "current_na", "db_file", "trial_number"]
    mode_col = "mechanistic_mode_signed_flux" if "mechanistic_mode_signed_flux" in phenotyped.columns else "buffering_phenotype"
    mode_vec = phenotyped.pivot_table(index=idx_cols, columns="window", values=mode_col, aggfunc="first").reset_index()
    mode_vec.columns = [f"mode_{c}" if c in WINDOWS else c for c in mode_vec.columns]
    total = phenotyped[phenotyped["window"] == "M_tot"].copy()
    add_cols = [c for c in ["buffering_phenotype", "temporal_recruitment_class", "gj_ionic_state_10_90", "final_n_by_S_tag", "final_surface_conductance_tag", "final_r_alpha_K_tag"] if c in total.columns]
    if add_cols:
        total = total[idx_cols + add_cols].drop_duplicates(idx_cols)
        mode_vec = mode_vec.merge(total, on=idx_cols, how="left")
    mode_dict: Dict[str, Dict[str, object]] = {}
    for _, r in mode_vec.iterrows():
        k = f"{r['experiment']}_{int(r['current_na'])}nA_trial{int(r['trial_number'])}"
        mode_dict[k] = {col: (None if pd.isna(r[col]) else r[col]) for col in mode_vec.columns if col not in idx_cols}
    mode_vec.to_csv(table_dir / "M_mode_vector_by_configuration.csv", index=False)
    with open(table_dir / "M_mode_dictionary.json", "w") as f:
        json.dump(mode_dict, f, indent=2, allow_nan=True)
    return mode_vec, mode_dict


def build_parameter_range_summary(features: pd.DataFrame, search_spaces: Optional[pd.DataFrame], table_dir: Path) -> pd.DataFrame:
    accepted = features[features.get("good_enough", False).astype(bool)].copy() if "good_enough" in features.columns else features.copy()
    param_cols = [c for c in ORDERED_PARAM_KEYS + ["Pkgap", "wa_over_wo", "drive_eps_x_deltaKbath", "drive_eps_x_Kbath"] if c in features.columns]
    rows = []
    for (exp, cur, db), g_all in features.groupby(["experiment", "current_na", "db_file"], dropna=False):
        g_acc = accepted[(accepted["experiment"] == exp) & (accepted["current_na"] == cur) & (accepted["db_file"] == db)]
        ss = None
        if search_spaces is not None and not search_spaces.empty:
            ss = search_spaces[(search_spaces.get("experiment") == exp) & (search_spaces.get("current_na") == cur) & (search_spaces.get("db_file") == db)]
        for col in param_cols:
            all_stats = numeric_summary_dict(g_all[col]) if col in g_all.columns else {}
            acc_stats = numeric_summary_dict(g_acc[col]) if col in g_acc.columns and not g_acc.empty else {}
            ss_row = ss[ss["param_name"] == col].iloc[0].to_dict() if ss is not None and not ss.empty and col in set(ss["param_name"]) else {}
            rows.append({
                "experiment": exp, "current_na": cur, "db_file": db, "parameter": col,
                "search_low": ss_row.get("low", np.nan), "search_high": ss_row.get("high", np.nan),
                "top_min": all_stats.get("min", np.nan), "top_max": all_stats.get("max", np.nan),
                "accepted_min": acc_stats.get("min", np.nan), "accepted_max": acc_stats.get("max", np.nan),
                "accepted_median": acc_stats.get("median", np.nan), "accepted_count": acc_stats.get("count", 0),
            })
    out = pd.DataFrame(rows)
    out.to_csv(table_dir / "parameter_range_constraints_before_after.csv", index=False)
    return out


def plot_parameter_range_constraints(range_df: pd.DataFrame, out_dir: Path, max_params: int = 20) -> None:
    if range_df.empty:
        return
    plot_dir = out_dir / "plots" / "parameters"
    plot_dir.mkdir(parents=True, exist_ok=True)
    g = range_df.copy()
    g["search_span"] = pd.to_numeric(g["search_high"], errors="coerce") - pd.to_numeric(g["search_low"], errors="coerce")
    g["accepted_span"] = pd.to_numeric(g["accepted_max"], errors="coerce") - pd.to_numeric(g["accepted_min"], errors="coerce")
    g["accepted_over_search_span"] = g["accepted_span"] / g["search_span"].replace(0, np.nan)
    agg = g.groupby("parameter")["accepted_over_search_span"].median().replace([np.inf, -np.inf], np.nan).dropna().sort_values().head(max_params)
    if agg.empty:
        return
    plt.figure(figsize=(7.2, max(4.0, 0.28 * len(agg))))
    plt.barh(range(len(agg)), agg.values)
    plt.yticks(range(len(agg)), agg.index)
    plt.xlabel("median accepted span / Optuna search span")
    plt.title("Parameter constraints after filtering")
    plt.tight_layout()
    plt.savefig(plot_dir / "accepted_over_search_span_by_parameter.png", dpi=180)
    plt.close()


def plot_signed_mode_quadrants(df: pd.DataFrame, out_dir: Path) -> None:
    if df.empty or "D_F_log10_local_load_over_spatial_export" not in df.columns:
        return
    plot_dir = out_dir / "plots" / "modes"
    plot_dir.mkdir(parents=True, exist_ok=True)
    for window, g in df.groupby("window"):
        g = g.replace([np.inf, -np.inf], np.nan).dropna(subset=["D_F_log10_local_load_over_spatial_export", "D_I_elec_log10_local_current_over_gap_current"])
        if g.empty:
            continue
        plt.figure(figsize=(7.0, 5.8))
        for mode, gp in g.groupby("mechanistic_mode_signed_flux"):
            plt.scatter(gp["D_F_log10_local_load_over_spatial_export"], gp["D_I_elec_log10_local_current_over_gap_current"], s=20, alpha=0.70, label=mode)
        theta = math.log10(DEFAULT_DOMINANCE_MARGIN)
        plt.axvline(theta, linestyle="--", linewidth=1)
        plt.axvline(-theta, linestyle="--", linewidth=1)
        plt.axhline(theta, linestyle="--", linewidth=1)
        plt.axhline(-theta, linestyle="--", linewidth=1)
        plt.xlabel("D_F = log10(local K load / spatial export)")
        plt.ylabel("D_I = log10(local K current / gap current)")
        plt.title(f"Corrected signed-flux mode plane - {window}")
        plt.legend(fontsize=7, loc="best")
        plt.tight_layout()
        plt.savefig(plot_dir / f"signed_flux_mode_quadrants_{window}.png", dpi=180)
        plt.close()


def plot_hidden_overlays_for_accepted(features: pd.DataFrame, out_dir: Path, sim_dt_ms: float, max_per_sweep: int = 8) -> None:
    if features.empty:
        return
    plot_dir = out_dir / "plots" / "hidden_overlays"
    plot_dir.mkdir(parents=True, exist_ok=True)
    accepted = features[features.get("good_enough", False).astype(bool)].copy() if "good_enough" in features.columns else features.copy()
    if accepted.empty:
        return
    signals_hidden = ["Ko", "Ka", "DKt", "Ks", "Kg", "gate_activation"]
    signals_currents = ["I_Kir", "I_k", "I_kgap", "I_l", "I_total_net"]
    for (exp, cur, db), g in accepted.groupby(["experiment", "current_na", "db_file"], dropna=False):
        g = g.sort_values("objective").head(max_per_sweep)
        if g.empty:
            continue
        fixed = {}
        try:
            db_path = Path(str(g.iloc[0].get("_db_path", "")))
            if db_path.exists():
                fixed = load_fixed_params_from_db(db_path)
                study_name = infer_study_name_from_db(db_path)
            else:
                study_name = str(g.iloc[0].get("study_name", f"{exp}_{cur}nA"))
        except Exception:
            study_name = f"{exp}_{cur}nA"
        context = make_context(str(exp), int(cur), study_name, "", "default", sim_dt_ms, fixed)
        details_list = []
        for _, row in g.iterrows():
            try:
                _, details, _ = simulate_trial(row, context)
                details_list.append((int(row["trial_number"]), details))
            except Exception:
                continue
        if not details_list:
            continue
        for signals, kind in [(signals_hidden, "hidden_states"), (signals_currents, "currents")]:
            n = len(signals)
            ncols = 2
            nrows = int(math.ceil(n / ncols))
            fig, axes = plt.subplots(nrows, ncols, figsize=(10.0, 2.6 * nrows), squeeze=False)
            for ax, signal in zip(axes.ravel(), signals):
                for trial_number, details in details_list:
                    y = details[signal] if signal in details else None
                    if y is not None:
                        ax.plot(details["t_s"], y, linewidth=0.8, alpha=0.55)
                ax.set_title(signal)
                ax.set_xlabel("time (s)")
            for ax in axes.ravel()[n:]:
                ax.axis("off")
            fig.suptitle(f"{kind}: {exp} {cur} nA accepted top {len(details_list)}")
            fig.tight_layout()
            fig.savefig(plot_dir / f"{kind}_{exp}_{int(cur)}nA.png", dpi=160)
            plt.close(fig)


def write_measure_registry(table_dir: Path) -> pd.DataFrame:
    df = pd.DataFrame(MEASURE_REGISTRY_ROWS)
    df.to_csv(table_dir / "measure_registry_status.csv", index=False)
    return df


def run_pipeline(args: argparse.Namespace) -> Dict[str, object]:
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    table_dir = out_dir / "tables"
    table_dir.mkdir(exist_ok=True)
    data_dir = ensure_data_dir(Path(args.data_zip) if args.data_zip else None, Path(args.data_dir) if args.data_dir else None, out_dir)
    threshold_df = None
    if args.threshold_csv and Path(args.threshold_csv).exists():
        threshold_df = pd.read_csv(args.threshold_csv)

    experiments = set(parse_csv_list(args.experiments, str)) if args.experiments else {"CONTROL", "MFA", "BARIUM"}
    currents = set(parse_csv_list(args.currents, int)) if args.currents else {50, 75, 100, 125, 150, 175}
    db_files = []
    for exp in experiments:
        for cur in currents:
            p = data_dir / f"{exp}_{cur}nA.db"
            if p.exists():
                db_files.append(p)
    if not db_files:
        raise FileNotFoundError(f"No DB files found in {data_dir}")

    all_feature_rows = []
    all_score_rows = []
    all_failures = []
    all_search_spaces = []

    for db_path in sorted(db_files):
        study_name = infer_study_name_from_db(db_path)
        experiment, current, target_mean_mode, objective_loss_type = infer_metadata_from_name(study_name, db_path)
        fixed = load_fixed_params_from_db(db_path)
        context = make_context(experiment, current, study_name, objective_loss_type, target_mean_mode, args.sim_dt_ms, fixed)
        trials = load_trials_from_db(db_path, include_penalty=args.include_penalty_trials)
        if trials.empty:
            continue
        trials["_db_path"] = str(db_path)
        n_use = len(trials) if args.top_n == 0 else min(args.top_n, len(trials))
        trials = trials.head(n_use).copy()
        print(f"{db_path.name}: simulating {len(trials)} trials at dt={args.sim_dt_ms} ms")

        ss = load_search_space_from_db(db_path)
        if not ss.empty:
            ss["db_file"] = db_path.name
            ss["experiment"] = experiment
            ss["current_na"] = current
            all_search_spaces.append(ss)

        for i, (_, row) in enumerate(trials.iterrows(), start=1):
            try:
                z, details, paramdict = simulate_trial(row, context)
                onset_s = context.windows_s["M_rise"][0]
                offset_s = context.windows_s["M_rise"][1]
                f_v = extract_trace_features(details["t_s"], details["Va"], onset_s, offset_s)
                f_ko = extract_trace_features(details["t_s"], details["Ko"], onset_s, offset_s)
                feat = {
                    "db_file": db_path.name,
                    "_db_path": str(db_path),
                    "study_name": study_name,
                    "experiment": experiment,
                    "current_na": current,
                    "trial_id": int(row["trial_id"]),
                    "trial_number": int(row["trial_number"]),
                    "objective": safe_float(row["objective"]),
                }
                for k in ORDERED_PARAM_KEYS:
                    if k in row:
                        feat[k] = row[k]
                feat.update({f"V_{k}": v for k, v in f_v.items()})
                # Duplicate Vm features without prefix for threshold CSV compatibility.
                feat.update(f_v)
                feat.update({f"Ko_{k}": v for k, v in f_ko.items()})
                feat.update(derived_param_columns(row))
                # Hidden sanity values.
                feat.update({
                    "hidden_Ko_min": float(np.nanmin(details["Ko"])),
                    "hidden_Ko_max": float(np.nanmax(details["Ko"])),
                    "hidden_Ka_min": float(np.nanmin(details["Ka"])),
                    "hidden_Ka_max": float(np.nanmax(details["Ka"])),
                    "hidden_Va_min": float(np.nanmin(details["Va"])),
                    "hidden_Va_max": float(np.nanmax(details["Va"])),
                    "hidden_max_abs_current": float(np.nanmax(np.abs(np.vstack([details["I_Kir"], details["I_kgap"], details["I_l"]])))),
                    "hidden_sanity_pass": bool(np.isfinite(z).all() and np.nanmin(details["Ko"]) > 0 and np.nanmax(details["Ko"]) < args.max_biological_Ko),
                })
                all_feature_rows.append(feat)
                score_rows = compute_window_scores(row, details, paramdict, context, dominance_margin=getattr(args, "dominance_margin", DEFAULT_DOMINANCE_MARGIN))
                for sr in score_rows:
                    sr["hidden_sanity_pass"] = feat["hidden_sanity_pass"]
                all_score_rows.extend(score_rows)
            except Exception as exc:
                all_failures.append({"db_file": db_path.name, "experiment": experiment, "current_na": current, "trial_number": int(row.get("trial_number", -1)), "error": str(exc)})
            if args.progress_every and i % args.progress_every == 0:
                print(f"  {db_path.name}: {i}/{len(trials)}")

    features = pd.DataFrame(all_feature_rows)
    scores = pd.DataFrame(all_score_rows)
    failures = pd.DataFrame(all_failures)
    if features.empty or scores.empty:
        raise RuntimeError("No successful simulations.")

    # Good-enough accepted set. If no threshold CSV is supplied, this marks top-N as provisional accepted.
    accepted_feature_tables = []
    for (exp, cur), g in features.groupby(["experiment", "current_na"]):
        sweep = CURRENT_DICT_COLUMNS[str(int(cur))]
        pass_table = build_feature_pass_table(g, threshold_df, sweep, group=args.threshold_group, mode=args.threshold_mode, min_pass_fraction=args.min_pass_fraction)
        accepted_feature_tables.append(pass_table)
    features = pd.concat(accepted_feature_tables, ignore_index=True)
    if args.require_hidden_sanity:
        features["good_enough"] = features["good_enough"] & features["hidden_sanity_pass"]

    # Optional fallback: if a threshold CSV is present but no trial passes for a sweep,
    # keep a clearly labelled objective-ranked provisional subset rather than returning an empty analysis.
    if threshold_df is not None and getattr(args, "auto_objective_fallback_if_empty", True):
        for (exp, cur), g in features.groupby(["experiment", "current_na"], dropna=False):
            if bool(g["good_enough"].any()):
                continue
            eligible = g[g["hidden_sanity_pass"]].copy() if getattr(args, "require_hidden_sanity", True) else g.copy()
            if eligible.empty:
                eligible = g.copy()
            n_fb = min(len(eligible), max(int(getattr(args, "fallback_min_n", 10)), int(math.ceil(len(eligible) * float(getattr(args, "fallback_fraction", 0.10))))))
            fb_idx = eligible.sort_values(["objective", "trial_number"]).head(n_fb).index
            features.loc[fb_idx, "good_enough"] = True
            features.loc[fb_idx, "acceptance_source"] = "objective_fallback_threshold_empty"

    key_cols = ["db_file", "trial_number"]
    accepted_keys = features.loc[features["good_enough"], key_cols].drop_duplicates()
    scores = scores.merge(accepted_keys.assign(good_enough=True), on=key_cols, how="left")
    scores["good_enough"] = scores["good_enough"].fillna(False)
    accepted_scores = scores[scores["good_enough"]].copy()

    # Phenotype classification.
    phenotyped, bin_thresholds = add_phenotypes(
        accepted_scores,
        bin_scope=args.bin_scope,
        low_q=args.low_quantile,
        high_q=args.high_quantile,
        mfa_remaining_fraction=args.mfa_remaining_fraction,
    )

    # Save tables.
    features.to_csv(table_dir / "all_top_trial_features_and_acceptance.csv", index=False)
    scores.to_csv(table_dir / "all_top_trial_mechanistic_scores_all_windows.csv", index=False)
    accepted_scores.to_csv(table_dir / "accepted_mechanistic_scores_prephenotype.csv", index=False)
    phenotyped.to_csv(table_dir / "accepted_mechanistic_scores_with_phenotypes.csv", index=False)
    bin_thresholds.to_csv(table_dir / "quantile_thresholds_used_for_bins.csv", index=False)
    if not failures.empty:
        failures.to_csv(table_dir / "failed_simulations.csv", index=False)
    search_spaces_df = pd.concat(all_search_spaces, ignore_index=True) if all_search_spaces else pd.DataFrame()
    if not search_spaces_df.empty:
        search_spaces_df.to_csv(table_dir / "optuna_search_spaces.csv", index=False)

    # Unified notebook outputs.
    measure_registry_df = write_measure_registry(table_dir)
    Fv, Fk, feature_variability_table = build_feature_variability_outputs(features, table_dir)
    mode_vector_df, mode_dictionary = build_mode_vector_outputs(phenotyped, table_dir)
    range_summary_df = build_parameter_range_summary(features, search_spaces_df, table_dir)

    counts = phenotyped.groupby(["experiment", "current_na", "window", "buffering_phenotype"]).size().reset_index(name="n")
    counts.to_csv(table_dir / "phenotype_counts_by_experiment_current_window.csv", index=False)
    gj_counts = phenotyped.groupby(["experiment", "current_na", "window", "gj_ionic_state_10_90"]).size().reset_index(name="n")
    gj_counts.to_csv(table_dir / "open_closed_gj_counts_by_experiment_current_window.csv", index=False)
    numeric_summary = phenotyped.groupby(["experiment", "current_na", "window"])[[
        "dKs_activation_score", "sigmoid_activation_mean", "long_range_distribution_fraction",
        "voltage_coupling_score", "kir_current_score", "ionic_coupling_score", "Ko_peak",
        "Ko_auc_above_baseline", "source_clearance_ratio", "buffering_efficiency_index",
        "predicted_MFA_sensitivity_score", "predicted_Ba_sensitivity_score", "predicted_pump_activity_proxy",
        "gs", "gs_x_Pkgap", "gs_x_d", "pump_drive_proxy_Ko_AUC", "pump_recovery_proxy_Ko_clearance", "pump_undershoot_proxy_mM",
    ]].agg(["count", "mean", "median", "std", "min", "max"]).reset_index()
    numeric_summary.columns = ["_".join([str(c) for c in tup if str(c) != ""]) for tup in numeric_summary.columns]
    numeric_summary.to_csv(table_dir / "numeric_score_summary_by_experiment_current_window.csv", index=False)
    add_mfa_control_contrasts(phenotyped, table_dir)

    # Phenotype dictionary for downstream use.
    pheno_dict = {}
    for _, r in phenotyped.iterrows():
        k = f"{r['experiment']}_{int(r['current_na'])}nA_trial{int(r['trial_number'])}_{r['window']}"
        pheno_dict[k] = {
            "phenotype": r["buffering_phenotype"],
            "gj_ionic_state_10_90": r["gj_ionic_state_10_90"],
            "dKs_activation_score": safe_float(r["dKs_activation_score"]),
            "long_range_distribution_fraction": safe_float(r["long_range_distribution_fraction"]),
            "kir_current_score": safe_float(r["kir_current_score"]),
            "voltage_coupling_score": safe_float(r["voltage_coupling_score"]),
            "Pkgap": safe_float(r["Pkgap"]),
            "gki": safe_float(r["gki"]),
            "d": safe_float(r["d"]),
            "predicted_MFA_sensitivity_score": safe_float(r["predicted_MFA_sensitivity_score"]),
            "predicted_Ba_sensitivity_score": safe_float(r["predicted_Ba_sensitivity_score"]),
            "predicted_pump_activity_proxy": safe_float(r.get("predicted_pump_activity_proxy")),
            "gs": safe_float(r.get("gs")),
            "gs_x_Pkgap": safe_float(r.get("gs_x_Pkgap")),
            "pump_drive_proxy_Ko_AUC": safe_float(r.get("pump_drive_proxy_Ko_AUC")),
            "pump_undershoot_proxy_mM": safe_float(r.get("pump_undershoot_proxy_mM")),
        }
    with open(table_dir / "phenotype_dictionary.json", "w") as f:
        json.dump(pheno_dict, f, indent=2, allow_nan=True)

    plot_activation_histograms(phenotyped, out_dir)
    plot_mode_space(phenotyped, out_dir)
    plot_signed_mode_quadrants(phenotyped, out_dir)
    plot_parameter_projection(phenotyped, out_dir)
    plot_parameter_range_constraints(range_summary_df, out_dir)
    if getattr(args, "make_hidden_overlays", True):
        plot_hidden_overlays_for_accepted(features, out_dir, args.sim_dt_ms, max_per_sweep=int(getattr(args, "max_overlay_per_sweep", 8)))

    summary = {
        "data_dir": str(data_dir),
        "n_db_files": len(db_files),
        "n_successful_trials": int(features.shape[0]),
        "n_accepted_trials_unique": int(features.loc[features["good_enough"], ["db_file", "trial_number"]].drop_duplicates().shape[0]),
        "n_accepted_window_rows": int(phenotyped.shape[0]),
        "threshold_source": str(args.threshold_csv) if threshold_df is not None else "none_top_n_provisional",
        "sim_dt_ms": args.sim_dt_ms,
        "top_n": args.top_n,
        "open_closed_thresholds": {"closed": "dKs_activation_score < 0.10", "open": "dKs_activation_score > 0.90"},
        "bin_scope": args.bin_scope,
        "low_quantile": args.low_quantile,
        "high_quantile": args.high_quantile,
        "mfa_remaining_fraction": args.mfa_remaining_fraction,
        "dominance_margin": getattr(args, "dominance_margin", DEFAULT_DOMINANCE_MARGIN),
        "n_Fv_features": len(Fv),
        "n_Fk_features": len(Fk),
        "n_mode_vectors": int(mode_vector_df.shape[0]) if isinstance(mode_vector_df, pd.DataFrame) else 0,
    }
    with open(out_dir / "run_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    readme_lines = [
        "Buffering phenotype pipeline outputs",
        "====================================",
        "",
        f"DB files processed: {len(db_files)}",
        f"Successful trial simulations: {features.shape[0]}",
        f"Accepted unique configurations: {summary['n_accepted_trials_unique']}",
        f"Accepted window rows: {phenotyped.shape[0]}",
        f"Threshold source: {summary['threshold_source']}",
        "",
        "Core activation definition:",
        "dKs_activation_score = integral(abs(dKs_actual)) / integral(abs(dKs_if_gate_fully_open))",
        "closed_low_redistribution: < 0.10; open_long_range_redistribution: > 0.90; intermediate otherwise.",
        "",
        "Main files:",
        "tables/accepted_mechanistic_scores_with_phenotypes.csv",
        "tables/phenotype_counts_by_experiment_current_window.csv",
        "tables/open_closed_gj_counts_by_experiment_current_window.csv",
        "tables/numeric_score_summary_by_experiment_current_window.csv",
        "tables/phenotype_dictionary.json",
        "tables/mfa_control_contrast_by_current_window.csv",
        "tables/mfa_control_phenotype_shift_by_current_window.csv",
        "tables/gs_pkgap_pump_ranges_by_experiment_current_window.csv",
    ]
    with open(out_dir / "README_outputs.txt", "w") as f:
        f.write("\n".join(readme_lines) + "\n")
    print("\n".join(readme_lines[:10]))
    print(f"Outputs written to: {out_dir}")

    if args.make_zip:
        zip_base = out_dir.with_suffix("")
        shutil.make_archive(str(zip_base), "zip", out_dir)
        print(f"Zip written: {zip_base}.zip")

    return {
        "features": features,
        "scores": scores,
        "accepted_scores": accepted_scores,
        "phenotyped": phenotyped,
        "bin_thresholds": bin_thresholds,
        "counts": counts,
        "gj_counts": gj_counts,
        "numeric_summary": numeric_summary,
        "feature_variability": feature_variability_table,
        "Fv": Fv,
        "Fk": Fk,
        "mode_vector": mode_vector_df,
        "mode_dictionary": mode_dictionary,
        "range_summary": range_summary_df,
        "measure_registry": measure_registry_df,
        "summary": summary,
        "out_dir": out_dir,
    }



## Configure the run

Upload `data.zip` to `/content/data.zip` in Colab, or edit `DATA_ZIP`. The default smoke profile extracts only the requested SQLite database files from the zip, which avoids expanding the full archive. Switch `RUN_PROFILE` to `full_light` or `full` for broader characterization.

In [2]:
from types import SimpleNamespace
from pathlib import Path
import shutil
import zipfile
import pandas as pd

# -----------------------------------------------------------------------------
# User configuration
# -----------------------------------------------------------------------------
# Options: "smoke", "full_light", "full"
RUN_PROFILE = "smoke"

# The notebook auto-detects a local upload in common Colab paths.
DATA_ZIP_CANDIDATES = [
    Path("/content/data.zip"),
    Path("/content/data(3).zip"),
    Path("/mnt/data/data(3).zip"),
    Path("data.zip"),
]
DATA_ZIP = next((p for p in DATA_ZIP_CANDIDATES if p.exists()), DATA_ZIP_CANDIDATES[0])

# Set this to an already extracted folder containing CONTROL_50nA.db etc. to skip zip extraction.
# Leave as None to selectively extract only the DBs required by the selected profile.
DATA_DIR_OVERRIDE = None

THRESHOLD_CSV_CANDIDATES = [
    Path("/content/threshold_for_good_enough_fits.csv"),
    Path("/mnt/data/threshold_for_good_enough_fits.csv"),
    Path("threshold_for_good_enough_fits.csv"),
]
THRESHOLD_CSV = next((p for p in THRESHOLD_CSV_CANDIDATES if p.exists()), None)

OUT_BASE = Path("/content/astro_buffering_unified_outputs") if Path("/content").exists() else Path("/mnt/data/astro_buffering_unified_outputs")
OUT_DIR = OUT_BASE / RUN_PROFILE
CLEAN_OUTPUT_DIR = True  # prevents stale files from earlier partial runs; set False to reuse outputs

PROFILES = {
    "smoke": {
        # Minimal fast path for checking that the notebook and database parser work.
        # Use full_light or full for the complete characterization.
        "experiments": "CONTROL",
        "currents": "50",
        "top_n": 2,
        "sim_dt_ms": 100.0,
        "make_hidden_overlays": True,
        "max_overlay_per_sweep": 1,
    },
    "full_light": {
        "experiments": "CONTROL,MFA,BARIUM",
        "currents": "50,75,100,125,150,175",
        "top_n": 30,
        "sim_dt_ms": 20.0,
        "make_hidden_overlays": True,
        "max_overlay_per_sweep": 4,
    },
    "full": {
        "experiments": "CONTROL,MFA,BARIUM",
        "currents": "50,75,100,125,150,175",
        "top_n": 300,
        "sim_dt_ms": 5.0,
        "make_hidden_overlays": True,
        "max_overlay_per_sweep": 8,
    },
}
profile = PROFILES[RUN_PROFILE]

if CLEAN_OUTPUT_DIR and OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

if DATA_DIR_OVERRIDE is not None:
    DATA_DIR = Path(DATA_DIR_OVERRIDE)
else:
    if not DATA_ZIP.exists():
        raise FileNotFoundError(
            f"Could not find data.zip. Upload it to Colab as /content/data.zip or edit DATA_ZIP. Current target: {DATA_ZIP}"
        )
    # Extract only the requested SQLite databases. The full zip expands to hundreds of MB,
    # while the analysis itself only needs the DB files for the chosen experiment/current set.
    selected_data_dir = OUT_DIR / "_selected_db_files"
    selected_data_dir.mkdir(parents=True, exist_ok=True)
    wanted = []
    for exp in parse_csv_list(profile["experiments"], str):
        for cur in parse_csv_list(profile["currents"], int):
            wanted.append(f"data/{str(exp).upper()}_{int(cur)}nA.db")
    with zipfile.ZipFile(DATA_ZIP, "r") as zf:
        names = zf.namelist()
        extracted = []
        for rel in wanted:
            matches = [n for n in names if n.endswith(rel)]
            if not matches:
                print("Missing in zip:", rel)
                continue
            src_name = matches[0]
            target = selected_data_dir / Path(rel).name
            with zf.open(src_name) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, length=1024 * 1024)
            extracted.append(target.name)
    DATA_DIR = selected_data_dir
    print("Extracted DB files:", extracted)

args = SimpleNamespace(
    data_zip=None,
    data_dir=str(DATA_DIR),
    threshold_csv=str(THRESHOLD_CSV) if THRESHOLD_CSV is not None else None,
    out=str(OUT_DIR),
    experiments=profile["experiments"],
    currents=profile["currents"],
    top_n=int(profile["top_n"]),
    sim_dt_ms=float(profile["sim_dt_ms"]),
    threshold_group="pooled",
    threshold_mode="min_max",
    min_pass_fraction=0.75,
    include_penalty_trials=False,
    require_hidden_sanity=True,
    max_biological_Ko=30.0,
    bin_scope="global",            # alternatives: window, experiment_window, experiment_current_window
    low_quantile=0.33,
    high_quantile=0.67,
    mfa_remaining_fraction=0.286,   # 1 - 0.714, if using 71.4% MFA block as an interpretive score
    dominance_margin=1.20,          # fold margin for strict/mixed mode labels
    auto_objective_fallback_if_empty=True,
    fallback_min_n=10,
    fallback_fraction=0.10,
    make_hidden_overlays=bool(profile["make_hidden_overlays"]),
    max_overlay_per_sweep=int(profile["max_overlay_per_sweep"]),
    progress_every=10,
    make_zip=False,
)

print("RUN_PROFILE:", RUN_PROFILE)
print("DATA_ZIP:", DATA_ZIP)
print("DATA_DIR:", DATA_DIR)
print("THRESHOLD_CSV:", THRESHOLD_CSV)
print("OUT_DIR:", OUT_DIR)
print("Experiments:", args.experiments)
print("Currents:", args.currents)
print("Top-N per DB:", args.top_n, "| sim_dt_ms:", args.sim_dt_ms)


Extracted DB files: ['CONTROL_50nA.db']
RUN_PROFILE: smoke
DATA_ZIP: /mnt/data/data(3).zip
DATA_DIR: /mnt/data/astro_buffering_unified_outputs/smoke/_selected_db_files
THRESHOLD_CSV: None
OUT_DIR: /mnt/data/astro_buffering_unified_outputs/smoke
Experiments: CONTROL
Currents: 50
Top-N per DB: 2 | sim_dt_ms: 100.0


## Run the pipeline

In [3]:

# Run the analysis. In the default smoke profile this should complete quickly.
results = run_pipeline(args)


CONTROL_50nA.db: simulating 2 trials at dt=100.0 ms


Buffering phenotype pipeline outputs

DB files processed: 1
Successful trial simulations: 2
Accepted unique configurations: 2
Accepted window rows: 8
Threshold source: none_top_n_provisional

Core activation definition:
Outputs written to: /mnt/data/astro_buffering_unified_outputs/smoke


## Inspect core outputs

In [4]:
from IPython.display import display

print("Run summary")
display(pd.Series(results["summary"]).to_frame("value"))

print("Measure registry: what is primary, secondary, superseded, or deprecated")
display(results["measure_registry"])

print("Accepted mechanistic scores with phenotypes")
preview_cols = [
    "experiment", "current_na", "trial_number", "window",
    "mechanistic_mode_signed_flux", "buffering_phenotype",
    "D_F_log10_local_load_over_spatial_export",
    "D_I_elec_log10_local_current_over_gap_current",
    "GJ_conductance_proxy_Pkgap",
    "functional_n_flux_proxy_dKs_activation",
    "isopotentiality_score_from_r",
]
preview_cols = [c for c in preview_cols if c in results["phenotyped"].columns]
display(results["phenotyped"][preview_cols].head(12))

print("M mode vector by configuration")
display(results["mode_vector"].head(12))

print("Fv/Fk feature variability")
display(results["feature_variability"].head(20))


Run summary


,value
data_dir,/mnt/data/astro_buffering_unified_outputs/smok...
n_db_files,1
n_successful_trials,2
n_accepted_trials_unique,2
n_accepted_window_rows,8
threshold_source,none_top_n_provisional
sim_dt_ms,100.0
top_n,2
open_closed_thresholds,"{'closed': 'dKs_activation_score < 0.10', 'ope..."
bin_scope,global


Measure registry: what is primary, secondary, superseded, or deprecated


,measure_family,measure,status,rationale
0,state reconstruction,"Ko, Ka, DKt, Ks, Kg, DKa, EKa, currents",primary,Hidden-variable reconstruction is retained for...
1,Vm/Ko features,Fv and Fk trace-feature dictionaries,primary,Parallel Vm and extracellular K shape features...
2,flux budget,"L=integral([dDKt/dt]+), S=integral([-dKs/dt]+)...",primary,Signed local load and spatial export resolve t...
3,mechanistic mode,D_F and D_I_elec with finite dominance margin,primary,Continuous axes are primary; strict/mixed labe...
4,recruitment,dKs_activation = integral(|dKs_actual|)/integr...,primary,Current-weighted recruitment is more mechanist...
5,Zhou/Ma mapping,d -> s_model; Pkgap=pk*d -> GJ conductance,primary,Corrected mapping; d is not syncytium size whe...
6,available surface,gs and alpha2 = gamma_s*Sigma_a/(w_a*F),primary,Interpreted as available spatial transfer surf...
7,functional syncytium,"chi_K, A_dKs, Ks, alpha2*chi_K, alpha2*A_dKs",primary,Dynamic recruited fraction/capacity; not anato...
8,isopotentiality proxy,"r_model=|Va-Vs|/(|EKa-Vs|+eps), 1/(1+r_model)",secondary,Useful reduced-model analogue because the ODE ...
9,pump proxy,Ko undershoot/recovery AUC proxies,secondary,"Retained as observable recovery descriptors, e..."


Accepted mechanistic scores with phenotypes


,experiment,current_na,trial_number,window,mechanistic_mode_signed_flux,buffering_phenotype,D_F_log10_local_load_over_spatial_export,D_I_elec_log10_local_current_over_gap_current,GJ_conductance_proxy_Pkgap,functional_n_flux_proxy_dKs_activation,isopotentiality_score_from_r
0,CONTROL,50,2067,M0,MIXED_LOCAL,largeAvailableSurface_highGJ_but_unrecruited,9.185547,-0.244733,0.000004,9.251221e-119,0.498631
1,CONTROL,50,2067,M_rise,MIXED_LOCAL,largeAvailableSurface_highGJ_but_unrecruited,10.243482,-0.274243,0.000004,4.062379e-60,0.501355
2,CONTROL,50,2067,M_decay,BALANCED_OR_WEAK,largeAvailableSurface_highGJ_but_unrecruited,6.818440,-0.057264,0.000004,2.831715e-59,0.497596
3,CONTROL,50,2067,M_tot,MIXED_LOCAL,largeAvailableSurface_highGJ_but_unrecruited,10.281353,-0.207201,0.000004,1.284852e-59,0.498675
4,CONTROL,50,2182,M0,BALANCED_OR_WEAK,smallN_lowS_lowGJ_GHK_like_local,9.307236,-0.021641,0.000002,5.303242e-110,0.498869
5,CONTROL,50,2182,M_rise,MIXED_LOCAL,bigN_lowGJ_large_range_weak_coupling,10.217870,-0.125227,0.000002,3.725199e-56,0.501049
6,CONTROL,50,2182,M_decay,BALANCED_OR_WEAK,bigN_lowGJ_large_range_weak_coupling,0.000000,0.135600,0.000002,2.223819e-55,0.498020
7,CONTROL,50,2182,M_tot,BALANCED_OR_WEAK,bigN_lowGJ_large_range_weak_coupling,10.269567,-0.037362,0.000002,1.039753e-55,0.498902


M mode vector by configuration


,experiment,current_na,db_file,trial_number,mode_M0,mode_M_decay,mode_M_rise,mode_M_tot,buffering_phenotype,temporal_recruitment_class,gj_ionic_state_10_90,final_n_by_S_tag,final_surface_conductance_tag,final_r_alpha_K_tag
0,CONTROL,50,CONTROL_50nA.db,2067,MIXED_LOCAL,BALANCED_OR_WEAK,MIXED_LOCAL,MIXED_LOCAL,largeAvailableSurface_highGJ_but_unrecruited,persistently_low_range_closed,closed_low_redistribution,nAvail_high__nRecruit_mid__nEnd_low__S_d_high_...,surface_high__recruitedSurface_mid__condOverSu...,r_mid__iso_mid__alpha_high__Ks_mid
1,CONTROL,50,CONTROL_50nA.db,2182,BALANCED_OR_WEAK,BALANCED_OR_WEAK,MIXED_LOCAL,BALANCED_OR_WEAK,bigN_lowGJ_large_range_weak_coupling,persistently_low_range_closed,closed_low_redistribution,nAvail_low__nRecruit_high__nEnd_high__S_d_low_...,surface_low__recruitedSurface_high__condOverSu...,r_low__iso_high__alpha_low__Ks_high


Fv/Fk feature variability


,dictionary,feature,count,min,q25,median,q75,max,range,mean,std
0,Fv,baseline_mV,2,-83.681224,-83.658118,-83.635012,-83.611907,-83.588801,0.092423,-83.635012,0.065353
1,Fv,peak_depolarization_mV,2,7.903807,7.921942,7.940077,7.958212,7.976347,0.072541,7.940077,0.051294
2,Fv,rise_slope_mV_per_s,2,7.976347,8.353402,8.730458,9.107513,9.484568,1.508221,8.730458,1.066473
3,Fv,rise_tau_s,2,0.427000,0.452000,0.477000,0.502000,0.527000,0.100000,0.477000,0.070711
4,Fv,plateau_slope_mV_per_s,2,-0.075909,-0.075207,-0.074504,-0.073802,-0.073100,0.002809,-0.074504,0.001986
5,Fv,decay_slope_mV_per_s,2,7.043368,7.294251,7.545133,7.796016,8.046898,1.003530,7.545133,0.709603
6,Fv,decay_tau_s,2,0.527000,0.527000,0.527000,0.527000,0.527000,0.000000,0.527000,0.000000
7,Fv,undershoot_magnitude_mV,2,0.908280,0.929117,0.949954,0.970790,0.991627,0.083347,0.949954,0.058935
8,Fv,return_slope_mV_per_s,2,0.037682,0.038063,0.038444,0.038825,0.039206,0.001524,0.038444,0.001077
9,Fk,baseline_mV,2,5.081986,5.082810,5.083635,5.084459,5.085284,0.003298,5.083635,0.002332


## Additional summaries

In [5]:

# Useful quick summaries after a run.
phenotyped = results["phenotyped"]

if not phenotyped.empty:
    print("Phenotype counts")
    display(
        phenotyped.groupby(["experiment", "current_na", "window", "buffering_phenotype"])
        .size()
        .reset_index(name="n")
        .sort_values(["experiment", "current_na", "window", "n"], ascending=[True, True, True, False])
        .head(50)
    )

    print("Signed flux mode counts")
    display(
        phenotyped.groupby(["experiment", "current_na", "window", "mechanistic_mode_signed_flux"])
        .size()
        .reset_index(name="n")
        .sort_values(["experiment", "current_na", "window", "n"], ascending=[True, True, True, False])
        .head(50)
    )

    selected_cols = [
        "experiment", "current_na", "window", "trial_number", "objective",
        "D_F_log10_local_load_over_spatial_export",
        "D_I_elec_log10_local_current_over_gap_current",
        "mechanistic_mode_signed_flux",
        "dKs_activation_score",
        "GJ_conductance_proxy_Pkgap",
        "n_available_proxy_gs",
        "functional_n_flux_proxy_dKs_activation",
        "buffering_phenotype",
    ]
    selected_cols = [c for c in selected_cols if c in phenotyped.columns]
    display(phenotyped[selected_cols].head(20))


Phenotype counts


,experiment,current_na,window,buffering_phenotype,n
0,CONTROL,50,M0,largeAvailableSurface_highGJ_but_unrecruited,1
1,CONTROL,50,M0,smallN_lowS_lowGJ_GHK_like_local,1
2,CONTROL,50,M_decay,bigN_lowGJ_large_range_weak_coupling,1
3,CONTROL,50,M_decay,largeAvailableSurface_highGJ_but_unrecruited,1
4,CONTROL,50,M_rise,bigN_lowGJ_large_range_weak_coupling,1
5,CONTROL,50,M_rise,largeAvailableSurface_highGJ_but_unrecruited,1
6,CONTROL,50,M_tot,bigN_lowGJ_large_range_weak_coupling,1
7,CONTROL,50,M_tot,largeAvailableSurface_highGJ_but_unrecruited,1


Signed flux mode counts


,experiment,current_na,window,mechanistic_mode_signed_flux,n
0,CONTROL,50,M0,BALANCED_OR_WEAK,1
1,CONTROL,50,M0,MIXED_LOCAL,1
2,CONTROL,50,M_decay,BALANCED_OR_WEAK,2
3,CONTROL,50,M_rise,MIXED_LOCAL,2
4,CONTROL,50,M_tot,BALANCED_OR_WEAK,1
5,CONTROL,50,M_tot,MIXED_LOCAL,1


,experiment,current_na,window,trial_number,objective,D_F_log10_local_load_over_spatial_export,D_I_elec_log10_local_current_over_gap_current,mechanistic_mode_signed_flux,dKs_activation_score,GJ_conductance_proxy_Pkgap,n_available_proxy_gs,functional_n_flux_proxy_dKs_activation,buffering_phenotype
0,CONTROL,50,M0,2067,21.502226,9.185547,-0.244733,MIXED_LOCAL,9.251221e-119,0.000004,10.972646,9.251221e-119,largeAvailableSurface_highGJ_but_unrecruited
1,CONTROL,50,M_rise,2067,21.502226,10.243482,-0.274243,MIXED_LOCAL,4.062379e-60,0.000004,10.972646,4.062379e-60,largeAvailableSurface_highGJ_but_unrecruited
2,CONTROL,50,M_decay,2067,21.502226,6.818440,-0.057264,BALANCED_OR_WEAK,2.831715e-59,0.000004,10.972646,2.831715e-59,largeAvailableSurface_highGJ_but_unrecruited
3,CONTROL,50,M_tot,2067,21.502226,10.281353,-0.207201,MIXED_LOCAL,1.284852e-59,0.000004,10.972646,1.284852e-59,largeAvailableSurface_highGJ_but_unrecruited
4,CONTROL,50,M0,2182,24.608913,9.307236,-0.021641,BALANCED_OR_WEAK,5.303242e-110,0.000002,10.894226,5.303242e-110,smallN_lowS_lowGJ_GHK_like_local
5,CONTROL,50,M_rise,2182,24.608913,10.217870,-0.125227,MIXED_LOCAL,3.725199e-56,0.000002,10.894226,3.725199e-56,bigN_lowGJ_large_range_weak_coupling
6,CONTROL,50,M_decay,2182,24.608913,0.000000,0.135600,BALANCED_OR_WEAK,2.223819e-55,0.000002,10.894226,2.223819e-55,bigN_lowGJ_large_range_weak_coupling
7,CONTROL,50,M_tot,2182,24.608913,10.269567,-0.037362,BALANCED_OR_WEAK,1.039753e-55,0.000002,10.894226,1.039753e-55,bigN_lowGJ_large_range_weak_coupling


## Optional zip/download helper

In [6]:

# Optional: zip outputs for download in Colab.
# Set CREATE_OUTPUT_ZIP=True and rerun this cell.
CREATE_OUTPUT_ZIP = False

if CREATE_OUTPUT_ZIP:
    import shutil
    zip_path = shutil.make_archive(str(Path(args.out)), "zip", Path(args.out))
    print("Created:", zip_path)
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception:
        print("Download helper is available only inside Colab. The zip is saved at:", zip_path)
